# SQLite Introduction

SQLite is a C-language library that implements a small, fast, self-contained, high-reliability, full-featured, SQL database engine. SQLite is the most used database engine in the world. SQLite is built into all mobile phones and most computers and comes bundled inside countless other applications that people use every day.
Source: https://www.sqlite.org/

## Load the magic function

In [1]:
%load_ext sql

## Create a database

In [2]:
%sql sqlite:///hospital.db

In [3]:
#Making a local connection with the created Databases
from sqlalchemy import create_engine
engine = create_engine("sqlite:///hospital.db")

## Create a table

In [4]:
from datetime import date


In [5]:
# dropping the user table if exists and then creating the user table

%%sql

DROP TABLE IF EXISTS User;

CREATE TABLE User (
    user_id INTEGER PRIMARY KEY,
    username TEXT NOT NULL UNIQUE,
    password TEXT NOT NULL,
    user_role TEXT NOT NULL CHECK(user_role IN ('admin', 'doctor', 'staff')),
    status TEXT NOT NULL DEFAULT 'active' CHECK(status IN ('active', 'inactive'))
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [6]:
#Run this cell to call the old version of SQL
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [7]:
# dropping the staff table if exists and then creating the staff table
%%sql

DROP TABLE IF EXISTS Staff;

CREATE TABLE Staff (
    staff_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL UNIQUE,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    phone TEXT NOT NULL,
    email TEXT NOT NULL,
    hire_date DATE NOT NULL,
    FOREIGN KEY (user_id) REFERENCES User(user_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [8]:
# dropping the doctor table if exists and then creating the doctor table
%%sql

DROP TABLE IF EXISTS Doctor;

CREATE TABLE Doctor (
    doctor_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL UNIQUE,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    phone TEXT NOT NULL,
    email TEXT,
    license_no TEXT NOT NULL UNIQUE,
    hire_date DATE NOT NULL,
    FOREIGN KEY (user_id) REFERENCES User(user_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [9]:
# dropping the patient table if exists and then creating the patient table
%%sql

DROP TABLE IF EXISTS Patient;

CREATE TABLE Patient (
    patient_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    date_of_birth DATE NOT NULL CHECK(date_of_birth > '1900-01-01' AND date_of_birth <= '2026-04-22'),
    gender TEXT NOT NULL,
    phone TEXT NOT NULL,
    email TEXT,
    address TEXT NOT NULL,
    registration_date DATE NOT NULL DEFAULT (DATE('now'))
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [10]:
# dropping the appointment table if exists and then creating the appointment table
%%sql

DROP TABLE IF EXISTS Appointment;

CREATE TABLE Appointment (
    appointment_id INTEGER PRIMARY KEY,
    doctor_id INTEGER NOT NULL,
    patient_id INTEGER NOT NULL,
    appointment_datetime DATETIME NOT NULL,
    status TEXT NOT NULL DEFAULT 'scheduled' CHECK(status IN ('scheduled', 'completed', 'cancelled', 'no_show')),
    reason_for_visit TEXT NOT NULL,
    notes TEXT NOT NULL,
    FOREIGN KEY (doctor_id)  REFERENCES Doctor(doctor_id),
    FOREIGN KEY (patient_id) REFERENCES Patient(patient_id),
    UNIQUE(doctor_id, appointment_datetime),
    UNIQUE(patient_id, appointment_datetime)
);


 * sqlite:///hospital.db
Done.
Done.


[]

In [11]:
# dropping the medical_record table if exists and then creating the medical_record table
%%sql

DROP TABLE IF EXISTS Medical_Record;

CREATE TABLE Medical_Record (
    record_id INTEGER PRIMARY KEY,
    appointment_id INTEGER NOT NULL UNIQUE,
    visit_date DATE NOT NULL,
    diagnosis TEXT NOT NULL,
    visit_notes TEXT NOT NULL,
    treatment_plan TEXT NOT NULL,
    created_date DATETIME NOT NULL DEFAULT (DATETIME('now')),
    last_modified_date DATETIME NOT NULL DEFAULT (DATETIME('now')) CHECK(last_modified_date >= created_date),
    lab_needed TEXT NOT NULL DEFAULT 'NO' CHECK(lab_needed IN ('YES', 'NO')),
    FOREIGN KEY (appointment_id) REFERENCES Appointment(appointment_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [12]:
# dropping the prescription table if exists and then creating the prescription table
%%sql

DROP TABLE IF EXISTS Prescription;

CREATE TABLE Prescription (
    prescription_id INTEGER PRIMARY KEY,
    record_id INTEGER NOT NULL,
    prescription_date DATE NOT NULL,
    medication_name TEXT NOT NULL,
    dosage TEXT NOT NULL,
    frequency TEXT NOT NULL,
    instructions TEXT NOT NULL,
    duration TEXT NOT NULL,
    FOREIGN KEY (record_id) REFERENCES Medical_Record(record_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [13]:
# dropping the billing table if exists and then creating the billing table
%%sql

DROP TABLE IF EXISTS Billing;

CREATE TABLE Billing (
    billing_id INTEGER PRIMARY KEY,
    appointment_id INTEGER NOT NULL UNIQUE,
    billing_date DATETIME NOT NULL DEFAULT (DATETIME('now')),
    total_amount REAL NOT NULL CHECK(total_amount >= 0),
    amount_paid REAL NOT NULL CHECK(amount_paid >= 0),
    amount_due REAL NOT NULL CHECK(amount_due >= 0),
    payment_status TEXT NOT NULL DEFAULT 'pending' CHECK(payment_status IN ('pending', 'paid', 'partial', 'overdue')),
    FOREIGN KEY (appointment_id) REFERENCES Appointment(appointment_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [14]:
# dropping the billing_item table if exists and then creating the billing_item table
%%sql

DROP TABLE IF EXISTS Billing_Item;

CREATE TABLE Billing_Item (
    item_id INTEGER PRIMARY KEY,
    billing_id INTEGER NOT NULL,
    service_type TEXT NOT NULL CHECK(service_type IN ('consultation', 'lab', 'medication', 'procedure')),
    description TEXT NOT NULL,
    unit_cost REAL NOT NULL CHECK(unit_cost > 0),
    quantity INTEGER NOT NULL CHECK(quantity > 0),
    total_cost REAL NOT NULL CHECK(total_cost > 0),
    FOREIGN KEY (billing_id) REFERENCES Billing(billing_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [15]:
# dropping the insurance_policy table if exists and then creating the insurance_policy table
%%sql

DROP TABLE IF EXISTS Insurance_Policy;


CREATE TABLE Insurance_Policy (
    policy_id INTEGER PRIMARY KEY,
    provider_name TEXT NOT NULL,
    policy_number TEXT NOT NULL UNIQUE,
    coverage_type TEXT NOT NULL,
    coverage_start_date DATE NOT NULL,
    coverage_end_date DATE NOT NULL CHECK(coverage_end_date > coverage_start_date),
    coverage_amount REAL NOT NULL CHECK(coverage_amount > 0)
);


 * sqlite:///hospital.db
Done.
Done.


[]

In [16]:
# dropping the patient_insurance table if exists and then creating the patient_insurance table
%%sql

DROP TABLE IF EXISTS Patient_Insurance;

CREATE TABLE Patient_Insurance (
    patient_id INTEGER NOT NULL,
    policy_id INTEGER NOT NULL,
    relationship_type TEXT,
    effective_date DATE,
    PRIMARY KEY (patient_id, policy_id),
    FOREIGN KEY (patient_id) REFERENCES Patient(patient_id),
    FOREIGN KEY (policy_id)  REFERENCES Insurance_Policy(policy_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [17]:
# dropping the insurance_claim table if exists and then creating the insurance_claim table
%%sql

DROP TABLE IF EXISTS Insurance_Claim;

CREATE TABLE Insurance_Claim (
    claim_id INTEGER PRIMARY KEY,
    policy_id INTEGER NOT NULL,
    billing_id INTEGER NOT NULL,
    claim_date DATETIME NOT NULL DEFAULT (DATETIME('now')),
    claim_amount REAL NOT NULL CHECK(claim_amount > 0),
    approval_date DATETIME NOT NULL,
    notes TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'draft' CHECK(status IN ('draft', 'submitted', 'under_review', 'approved', 'denied')),
    FOREIGN KEY (policy_id)  REFERENCES Insurance_Policy(policy_id),
    FOREIGN KEY (billing_id) REFERENCES Billing(billing_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [18]:
# dropping the lab_test table if exists and then creating the lab_test table
%%sql

DROP TABLE IF EXISTS Lab_Test;

CREATE TABLE Lab_Test (
    test_id INTEGER PRIMARY KEY,
    test_name TEXT NOT NULL UNIQUE,
    test_description TEXT NOT NULL,
    standard_cost REAL NOT NULL CHECK(standard_cost >= 0)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [19]:
# dropping the lab_report table if exists and then creating the lab_report table
%%sql

DROP TABLE IF EXISTS Lab_Report;

CREATE TABLE Lab_Report (
    report_id INTEGER PRIMARY KEY,
    record_id INTEGER NOT NULL,
    user_id INTEGER NOT NULL,
    test_id INTEGER NOT NULL,
    test_date DATE NOT NULL,
    result TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'pending' CHECK(status IN ('pending', 'completed', 'reviewed', 'archived')),
    report_file_path TEXT NOT NULL,
    notes TEXT NOT NULL,
    FOREIGN KEY (record_id) REFERENCES Medical_Record(record_id),
    FOREIGN KEY (user_id) REFERENCES User(user_id),
    FOREIGN KEY (test_id) REFERENCES Lab_Test(test_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [20]:
# dropping the specialization table if exists and then creating the specialization table
%%sql

DROP TABLE IF EXISTS Specialization;

CREATE TABLE Specialization (
    specialization_id INTEGER PRIMARY KEY,
    specialization_name TEXT NOT NULL UNIQUE,
    description TEXT NOT NULL
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [21]:
# dropping the doctor_specialization table if exists and then creating the doctor_specialization table
%%sql

DROP TABLE IF EXISTS Doctor_Specialization;

CREATE TABLE Doctor_Specialization (
    doctor_id INTEGER NOT NULL,
    specialization_id INTEGER NOT NULL,
    assigned_date DATE NOT NULL,
    PRIMARY KEY (doctor_id, specialization_id),
    FOREIGN KEY (doctor_id) REFERENCES Doctor(doctor_id),
    FOREIGN KEY (specialization_id) REFERENCES Specialization(specialization_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

In [22]:
# dropping the doctor_availability table if exists and then creating the doctor_availability table
%%sql

DROP TABLE IF EXISTS Doctor_Availability;

CREATE TABLE Doctor_Availability (
    availability_id INTEGER PRIMARY KEY,
    doctor_id INTEGER NOT NULL,
    slot_start_time DATETIME NOT NULL,
    slot_end_time DATETIME NOT NULL CHECK(slot_end_time > slot_start_time),
    is_available TEXT NOT NULL DEFAULT 'YES' CHECK(is_available IN ('YES', 'NO')),
    FOREIGN KEY (doctor_id) REFERENCES Doctor(doctor_id)
);

 * sqlite:///hospital.db
Done.
Done.


[]

## Insert (aka load) data  into the table

In [23]:
%%sql

--USER TABLE
INSERT INTO User (user_id, username, password, user_role, status) VALUES
(1, 'admin.johnson', 'hashed_pw_001', 'admin', 'active'),
(2, 'admin.williams', 'hashed_pw_002', 'admin', 'active'),
(3, 'admin.brown', 'hashed_pw_003', 'admin', 'active'),
(4, 'admin.jones', 'hashed_pw_004', 'admin', 'active'),
(5, 'admin.garcia', 'hashed_pw_005', 'admin', 'active'),
(6, 'admin.miller', 'hashed_pw_006', 'admin', 'active'),
(7, 'admin.davis', 'hashed_pw_007', 'admin', 'active'),
(8, 'admin.rodriguez', 'hashed_pw_008', 'admin', 'active'),
(9, 'admin.martinez', 'hashed_pw_009', 'admin', 'active'),
(10, 'admin.hernandez', 'hashed_pw_010', 'admin', 'active'),
(11, 'admin.lopez', 'hashed_pw_011', 'admin', 'active'),
(12, 'admin.gonzalez', 'hashed_pw_012', 'admin', 'active'),
(13, 'admin.wilson', 'hashed_pw_013', 'admin', 'active'),
(14, 'admin.anderson', 'hashed_pw_014', 'admin', 'active'),
(15, 'admin.thomas', 'hashed_pw_015', 'admin', 'active'),
(16, 'admin.taylor', 'hashed_pw_016', 'admin', 'active'),
(17, 'admin.moore', 'hashed_pw_017', 'admin', 'active'),
(18, 'admin.jackson', 'hashed_pw_018', 'admin', 'active'),
(19, 'admin.martin', 'hashed_pw_019', 'admin', 'active'),
(20, 'admin.lee', 'hashed_pw_020', 'admin', 'active'),
(21, 'admin.perez', 'hashed_pw_021', 'admin', 'active'),
(22, 'admin.thompson', 'hashed_pw_022', 'admin', 'active'),
(23, 'admin.white', 'hashed_pw_023', 'admin', 'active'),
(24, 'admin.harris', 'hashed_pw_024', 'admin', 'active'),
(25, 'admin.sanchez', 'hashed_pw_025', 'admin', 'active'),
(26, 'admin.clark', 'hashed_pw_026', 'admin', 'active'),
(27, 'admin.ramirez', 'hashed_pw_027', 'admin', 'active'),
(28, 'admin.lewis', 'hashed_pw_028', 'admin', 'active'),
(29, 'admin.robinson', 'hashed_pw_029', 'admin', 'inactive'),
(30, 'admin.walker', 'hashed_pw_030', 'admin', 'active'),
(31, 'dr.smith', 'hashed_pw_031', 'doctor', 'active'),
(32, 'dr.patel', 'hashed_pw_032', 'doctor', 'active'),
(33, 'dr.chen', 'hashed_pw_033', 'doctor', 'active'),
(34, 'dr.kim', 'hashed_pw_034', 'doctor', 'active'),
(35, 'dr.nguyen', 'hashed_pw_035', 'doctor', 'active'),
(36, 'dr.singh', 'hashed_pw_036', 'doctor', 'active'),
(37, 'dr.ahmed', 'hashed_pw_037', 'doctor', 'active'),
(38, 'dr.murphy', 'hashed_pw_038', 'doctor', 'active'),
(39, 'dr.oconnor', 'hashed_pw_039', 'doctor', 'active'),
(40, 'dr.rivera', 'hashed_pw_040', 'doctor', 'active'),
(41, 'dr.torres', 'hashed_pw_041', 'doctor', 'active'),
(42, 'dr.flores', 'hashed_pw_042', 'doctor', 'active'),
(43, 'dr.cooper', 'hashed_pw_043', 'doctor', 'active'),
(44, 'dr.bailey', 'hashed_pw_044', 'doctor', 'active'),
(45, 'dr.reed', 'hashed_pw_045', 'doctor', 'active'),
(46, 'dr.kelly', 'hashed_pw_046', 'doctor', 'active'),
(47, 'dr.howard', 'hashed_pw_047', 'doctor', 'active'),
(48, 'dr.ward', 'hashed_pw_048', 'doctor', 'active'),
(49, 'dr.cox', 'hashed_pw_049', 'doctor', 'active'),
(50, 'dr.diaz', 'hashed_pw_050', 'doctor', 'active'),
(51, 'dr.richardson', 'hashed_pw_051', 'doctor', 'active'),
(52, 'dr.wood', 'hashed_pw_052', 'doctor', 'active'),
(53, 'dr.watson', 'hashed_pw_053', 'doctor', 'active'),
(54, 'dr.brooks', 'hashed_pw_054', 'doctor', 'active'),
(55, 'dr.bennett', 'hashed_pw_055', 'doctor', 'active'),
(56, 'dr.gray', 'hashed_pw_056', 'doctor', 'active'),
(57, 'dr.mendoza', 'hashed_pw_057', 'doctor', 'active'),
(58, 'dr.ruiz', 'hashed_pw_058', 'doctor', 'active'),
(59, 'dr.hughes', 'hashed_pw_059', 'doctor', 'inactive'),
(60, 'dr.price', 'hashed_pw_060', 'doctor', 'active'),
(61, 'staff.adams', 'hashed_pw_061', 'staff', 'active'),
(62, 'staff.baker', 'hashed_pw_062', 'staff', 'active'),
(63, 'staff.carter', 'hashed_pw_063', 'staff', 'active'),
(64, 'staff.dixon', 'hashed_pw_064', 'staff', 'active'),
(65, 'staff.evans', 'hashed_pw_065', 'staff', 'active'),
(66, 'staff.foster', 'hashed_pw_066', 'staff', 'active'),
(67, 'staff.green', 'hashed_pw_067', 'staff', 'active'),
(68, 'staff.hill', 'hashed_pw_068', 'staff', 'active'),
(69, 'staff.jenkins', 'hashed_pw_069', 'staff', 'active'),
(70, 'staff.king', 'hashed_pw_070', 'staff', 'active'),
(71, 'staff.long', 'hashed_pw_071', 'staff', 'active'),
(72, 'staff.morgan', 'hashed_pw_072', 'staff', 'active'),
(73, 'staff.nelson', 'hashed_pw_073', 'staff', 'active'),
(74, 'staff.owens', 'hashed_pw_074', 'staff', 'active'),
(75, 'staff.parker', 'hashed_pw_075', 'staff', 'active'),
(76, 'staff.quinn', 'hashed_pw_076', 'staff', 'active'),
(77, 'staff.ross', 'hashed_pw_077', 'staff', 'active'),
(78, 'staff.scott', 'hashed_pw_078', 'staff', 'active'),
(79, 'staff.turner', 'hashed_pw_079', 'staff', 'active'),
(80, 'staff.powell', 'hashed_pw_080', 'staff', 'active'),
(81, 'staff.perry', 'hashed_pw_081', 'staff', 'active'),
(82, 'staff.butler', 'hashed_pw_082', 'staff', 'active'),
(83, 'staff.barnes', 'hashed_pw_083', 'staff', 'active'),
(84, 'staff.fisher', 'hashed_pw_084', 'staff', 'active'),
(85, 'staff.henderson', 'hashed_pw_085', 'staff', 'active'),
(86, 'staff.coleman', 'hashed_pw_086', 'staff', 'active'),
(87, 'staff.simmons', 'hashed_pw_087', 'staff', 'active'),
(88, 'staff.patterson', 'hashed_pw_088', 'staff', 'active'),
(89, 'staff.jordan', 'hashed_pw_089', 'staff', 'inactive'),
(90, 'staff.reynolds', 'hashed_pw_090', 'staff', 'active');


-- STAFF TABLE

INSERT INTO Staff (staff_id, user_id, first_name, last_name, phone, email, hire_date) VALUES
(1, 61, 'Emma', 'Adams', '555-0101', 'emma.adams@clinic.com', '2020-03-15'),
(2, 62, 'Oliver', 'Baker', '555-0102', 'oliver.baker@clinic.com', '2019-07-22'),
(3, 63, 'Sophia', 'Carter', '555-0103', 'sophia.carter@clinic.com', '2021-01-10'),
(4, 64, 'Liam', 'Dixon', '555-0104', 'liam.dixon@clinic.com', '2018-11-05'),
(5, 65, 'Ava', 'Evans', '555-0105', 'ava.evans@clinic.com', '2022-02-14'),
(6, 66, 'Noah', 'Foster', '555-0106', 'noah.foster@clinic.com', '2020-09-30'),
(7, 67, 'Isabella', 'Green', '555-0107', 'isabella.green@clinic.com', '2019-04-18'),
(8, 68, 'Ethan', 'Hill', '555-0108', 'ethan.hill@clinic.com', '2021-06-25'),
(9, 69, 'Mia', 'Jenkins', '555-0109', 'mia.jenkins@clinic.com', '2020-12-08'),
(10, 70, 'Mason', 'King', '555-0110', 'mason.king@clinic.com', '2018-08-16'),
(11, 71, 'Charlotte', 'Long', '555-0111', 'charlotte.long@clinic.com', '2022-05-03'),
(12, 72, 'Lucas', 'Morgan', '555-0112', 'lucas.morgan@clinic.com', '2019-10-11'),
(13, 73, 'Amelia', 'Nelson', '555-0113', 'amelia.nelson@clinic.com', '2021-03-27'),
(14, 74, 'Logan', 'Owens', '555-0114', 'logan.owens@clinic.com', '2020-07-19'),
(15, 75, 'Harper', 'Parker', '555-0115', 'harper.parker@clinic.com', '2018-12-01'),
(16, 76, 'Jackson', 'Quinn', '555-0116', 'jackson.quinn@clinic.com', '2022-08-22'),
(17, 77, 'Evelyn', 'Ross', '555-0117', 'evelyn.ross@clinic.com', '2019-02-14'),
(18, 78, 'Sebastian', 'Scott', '555-0118', 'sebastian.scott@clinic.com', '2021-11-06'),
(19, 79, 'Ella', 'Turner', '555-0119', 'ella.turner@clinic.com', '2020-04-29'),
(20, 80, 'Aiden', 'Powell', '555-0120', 'aiden.powell@clinic.com', '2019-09-17'),
(21, 81, 'Scarlett', 'Perry', '555-0121', 'scarlett.perry@clinic.com', '2022-01-10'),
(22, 82, 'Jack', 'Butler', '555-0122', 'jack.butler@clinic.com', '2018-05-23'),
(23, 83, 'Grace', 'Barnes', '555-0123', 'grace.barnes@clinic.com', '2021-07-15'),
(24, 84, 'Henry', 'Fisher', '555-0124', 'henry.fisher@clinic.com', '2020-10-08'),
(25, 85, 'Lily', 'Henderson', '555-0125', 'lily.henderson@clinic.com', '2019-03-30'),
(26, 86, 'Owen', 'Coleman', '555-0126', 'owen.coleman@clinic.com', '2022-06-12'),
(27, 87, 'Zoe', 'Simmons', '555-0127', 'zoe.simmons@clinic.com', '2018-09-04'),
(28, 88, 'Carter', 'Patterson', '555-0128', 'carter.patterson@clinic.com', '2021-12-20'),
(29, 89, 'Aria', 'Jordan', '555-0129', 'aria.jordan@clinic.com', '2020-02-05'),
(30, 90, 'Luke', 'Reynolds', '555-0130', 'luke.reynolds@clinic.com', '2019-08-28');

-- DOCTOR TABLE

INSERT INTO Doctor (doctor_id, user_id, first_name, last_name, phone, email, license_no, hire_date) VALUES
(1, 31, 'Robert', 'Smith', '555-0201', 'dr.smith@clinic.com', 'MD-2015-001234', '2015-06-01'),
(2, 32, 'Priya', 'Patel', '555-0202', 'dr.patel@clinic.com', 'MD-2016-002345', '2016-03-15'),
(3, 33, 'Wei', 'Chen', '555-0203', 'dr.chen@clinic.com', 'MD-2014-003456', '2014-09-10'),
(4, 34, 'Min', 'Kim', '555-0204', 'dr.kim@clinic.com', 'MD-2017-004567', '2017-01-20'),
(5, 35, 'Linh', 'Nguyen', '555-0205', 'dr.nguyen@clinic.com', 'MD-2015-005678', '2015-11-08'),
(6, 36, 'Rajesh', 'Singh', '555-0206', 'dr.singh@clinic.com', 'MD-2018-006789', '2018-04-12'),
(7, 37, 'Fatima', 'Ahmed', '555-0207', 'dr.ahmed@clinic.com', 'MD-2016-007890', '2016-07-25'),
(8, 38, 'Sean', 'Murphy', '555-0208', 'dr.murphy@clinic.com', 'MD-2019-008901', '2019-02-14'),
(9, 39, 'Patrick', 'OConnor', '555-0209', 'dr.oconnor@clinic.com', 'MD-2015-009012', '2015-08-30'),
(10, 40, 'Maria', 'Rivera', '555-0210', 'dr.rivera@clinic.com', 'MD-2020-010123', '2020-05-18'),
(11, 41, 'Carlos', 'Torres', '555-0211', 'dr.torres@clinic.com', 'MD-2017-011234', '2017-10-03'),
(12, 42, 'Ana', 'Flores', '555-0212', 'dr.flores@clinic.com', 'MD-2018-012345', '2018-12-11'),
(13, 43, 'James', 'Cooper', '555-0213', 'dr.cooper@clinic.com', 'MD-2016-013456', '2016-06-22'),
(14, 44, 'Sarah', 'Bailey', '555-0214', 'dr.bailey@clinic.com', 'MD-2019-014567', '2019-09-07'),
(15, 45, 'Michael', 'Reed', '555-0215', 'dr.reed@clinic.com', 'MD-2015-015678', '2015-03-29'),
(16, 46, 'Jennifer', 'Kelly', '555-0216', 'dr.kelly@clinic.com', 'MD-2021-016789', '2021-01-15'),
(17, 47, 'David', 'Howard', '555-0217', 'dr.howard@clinic.com', 'MD-2017-017890', '2017-04-08'),
(18, 48, 'Lisa', 'Ward', '555-0218', 'dr.ward@clinic.com', 'MD-2018-018901', '2018-11-19'),
(19, 49, 'Daniel', 'Cox', '555-0219', 'dr.cox@clinic.com', 'MD-2016-019012', '2016-02-26'),
(20, 50, 'Carmen', 'Diaz', '555-0220', 'dr.diaz@clinic.com', 'MD-2020-020123', '2020-07-13'),
(21, 51, 'Thomas', 'Richardson', '555-0221', 'dr.richardson@clinic.com', 'MD-2015-021234', '2015-12-05'),
(22, 52, 'Amanda', 'Wood', '555-0222', 'dr.wood@clinic.com', 'MD-2019-022345', '2019-05-21'),
(23, 53, 'Christopher', 'Watson', '555-0223', 'dr.watson@clinic.com', 'MD-2017-023456', '2017-08-14'),
(24, 54, 'Michelle', 'Brooks', '555-0224', 'dr.brooks@clinic.com', 'MD-2021-024567', '2021-03-02'),
(25, 55, 'Andrew', 'Bennett', '555-0225', 'dr.bennett@clinic.com', 'MD-2018-025678', '2018-10-27'),
(26, 56, 'Rachel', 'Gray', '555-0226', 'dr.gray@clinic.com', 'MD-2016-026789', '2016-01-09'),
(27, 57, 'Jorge', 'Mendoza', '555-0227', 'dr.mendoza@clinic.com', 'MD-2020-027890', '2020-06-16'),
(28, 58, 'Elena', 'Ruiz', '555-0228', 'dr.ruiz@clinic.com', 'MD-2019-028901', '2019-11-24'),
(29, 59, 'Kevin', 'Hughes', '555-0229', 'dr.hughes@clinic.com', 'MD-2015-029012', '2015-04-11'),
(30, 60, 'Victoria', 'Price', '555-0230', 'dr.price@clinic.com', 'MD-2022-030123', '2022-01-28');

-- PATIENT TABLE

INSERT INTO Patient (patient_id, first_name, last_name, date_of_birth, gender, phone, email, address, registration_date) VALUES
(1, 'Benjamin', 'Anderson', '1985-03-12', 'Male', '555-0301', 'b.anderson@email.com', '123 Oak Street, Dallas, TX 75201', '2023-01-15'),
(2, 'Emily', 'Martinez', '1992-07-22', 'Female', '555-0302', 'e.martinez@email.com', '456 Elm Avenue, Dallas, TX 75202', '2023-02-20'),
(3, 'Jacob', 'Thompson', '1978-11-05', 'Male', '555-0303', 'j.thompson@email.com', '789 Pine Road, Dallas, TX 75203', '2023-03-10'),
(4, 'Olivia', 'White', '1988-04-18', 'Female', '555-0304', 'o.white@email.com', '321 Maple Lane, Dallas, TX 75204', '2023-04-05'),
(5, 'William', 'Lopez', '1995-09-30', 'Male', '555-0305', 'w.lopez@email.com', '654 Birch Drive, Dallas, TX 75205', '2023-05-12'),
(6, 'Sophia', 'Lee', '1983-12-08', 'Female', '555-0306', 's.lee@email.com', '987 Cedar Court, Dallas, TX 75206', '2023-06-18'),
(7, 'James', 'Walker', '1990-02-14', 'Male', '555-0307', 'j.walker@email.com', '147 Spruce Street, Dallas, TX 75207', '2023-07-25'),
(8, 'Ava', 'Hall', '1987-06-25', 'Female', '555-0308', 'a.hall@email.com', '258 Willow Way, Dallas, TX 75208', '2023-08-30'),
(9, 'Michael', 'Allen', '1993-10-11', 'Male', '555-0309', 'm.allen@email.com', '369 Ash Boulevard, Dallas, TX 75209', '2023-09-14'),
(10, 'Isabella', 'Young', '1981-01-27', 'Female', '555-0310', 'i.young@email.com', '741 Cherry Circle, Dallas, TX 75210', '2023-10-22'),
(11, 'Alexander', 'King', '1996-05-03', 'Male', '555-0311', 'a.king@email.com', '852 Hickory Place, Dallas, TX 75211', '2023-11-08'),
(12, 'Mia', 'Wright', '1989-08-19', 'Female', '555-0312', 'm.wright@email.com', '963 Walnut Terrace, Dallas, TX 75212', '2023-12-15'),
(13, 'Ethan', 'Scott', '1984-03-29', 'Male', '555-0313', 'e.scott@email.com', '159 Poplar Avenue, Dallas, TX 75213', '2024-01-20'),
(14, 'Charlotte', 'Torres', '1991-07-15', 'Female', '555-0314', 'c.torres@email.com', '357 Magnolia Drive, Dallas, TX 75214', '2024-02-10'),
(15, 'Daniel', 'Nguyen', '1986-11-06', 'Male', '555-0315', 'd.nguyen@email.com', '468 Sycamore Lane, Dallas, TX 75215', '2024-03-05'),
(16, 'Amelia', 'Hill', '1994-02-26', 'Female', '555-0316', 'a.hill@email.com', '579 Redwood Road, Dallas, TX 75216', '2024-04-12'),
(17, 'Matthew', 'Green', '1982-06-12', 'Male', '555-0317', 'm.green@email.com', '680 Beech Street, Dallas, TX 75217', '2024-05-18'),
(18, 'Harper', 'Adams', '1997-09-17', 'Female', '555-0318', 'h.adams@email.com', '791 Juniper Court, Dallas, TX 75218', '2024-06-25'),
(19, 'Joseph', 'Baker', '1980-12-01', 'Male', '555-0319', 'j.baker@email.com', '802 Cypress Way, Dallas, TX 75219', '2024-07-30'),
(20, 'Evelyn', 'Gonzalez', '1998-04-22', 'Female', '555-0320', 'e.gonzalez@email.com', '913 Fir Place, Dallas, TX 75220', '2024-08-14'),
(21, 'David', 'Nelson', '1985-08-08', 'Male', '555-0321', 'd.nelson@email.com', '124 Laurel Avenue, Dallas, TX 75221', '2024-09-19'),
(22, 'Abigail', 'Carter', '1992-11-24', 'Female', '555-0322', 'a.carter@email.com', '235 Dogwood Drive, Dallas, TX 75222', '2024-10-25'),
(23, 'Christopher', 'Mitchell', '1979-03-10', 'Male', '555-0323', 'c.mitchell@email.com', '346 Elm Street, Dallas, TX 75223', '2024-11-30'),
(24, 'Elizabeth', 'Perez', '1990-07-16', 'Female', '555-0324', 'e.perez@email.com', '457 Oak Avenue, Dallas, TX 75224', '2024-12-15'),
(25, 'Andrew', 'Roberts', '1987-10-28', 'Male', '555-0325', 'a.roberts@email.com', '568 Pine Lane, Dallas, TX 75225', '2025-01-10'),
(26, 'Emily', 'Turner', '1995-01-13', 'Female', '555-0326', 'e.turner@email.com', '679 Maple Road, Dallas, TX 75226', '2025-02-14'),
(27, 'Joshua', 'Phillips', '1983-05-21', 'Male', '555-0327', 'j.phillips@email.com', '780 Birch Court, Dallas, TX 75227', '2025-03-20'),
(28, 'Madison', 'Campbell', '1991-08-05', 'Female', '555-0328', 'm.campbell@email.com', '891 Cedar Way, Dallas, TX 75228', '2025-04-05'),
(29, 'Ryan', 'Parker', '1988-12-19', 'Male', '555-0329', 'r.parker@email.com', '902 Spruce Place, Dallas, TX 75229', '2025-05-12'),
(30, 'Grace', 'Evans', '1996-02-07', 'Female', '555-0330', 'g.evans@email.com', '113 Willow Drive, Dallas, TX 75230', '2025-06-18');

-- APPOINTMENT TABLE

INSERT INTO Appointment (appointment_id, doctor_id, patient_id, appointment_datetime, status, reason_for_visit, notes) VALUES
(1, 1, 1, '2025-01-15 09:00:00', 'completed', 'Annual physical examination', 'Patient arrived on time, vitals normal'),
(2, 2, 2, '2025-01-16 10:30:00', 'completed', 'Follow-up for hypertension', 'Blood pressure improved since last visit'),
(3, 3, 3, '2025-01-17 14:00:00', 'completed', 'Persistent cough and fever', 'Symptoms started 5 days ago'),
(4, 4, 4, '2025-01-18 11:00:00', 'completed', 'Diabetes management consultation', 'Reviewing recent lab results'),
(5, 5, 5, '2025-01-19 15:30:00', 'completed', 'Sports injury - knee pain', 'Injured during basketball game'),
(6, 6, 6, '2025-01-20 09:30:00', 'completed', 'Prenatal checkup - 20 weeks', 'Second pregnancy, no complications'),
(7, 7, 7, '2025-01-21 13:00:00', 'completed', 'Skin rash evaluation', 'Rash appeared 2 weeks ago on arms'),
(8, 8, 8, '2025-01-22 16:00:00', 'completed', 'Migraine headaches consultation', 'Frequency increasing over past month'),
(9, 9, 9, '2025-01-23 10:00:00', 'completed', 'Chest pain assessment', 'Intermittent pain for 3 days'),
(10, 10, 10, '2025-01-24 14:30:00', 'completed', 'Depression and anxiety screening', 'Feeling overwhelmed lately'),
(11, 11, 11, '2025-01-25 11:30:00', 'completed', 'Lower back pain treatment', 'Pain started after moving furniture'),
(12, 12, 12, '2025-01-26 09:00:00', 'completed', 'Asthma management review', 'Recent increase in symptoms'),
(13, 13, 13, '2025-01-27 15:00:00', 'completed', 'Stomach pain and nausea', 'Symptoms for 2 days'),
(14, 14, 14, '2025-01-28 10:30:00', 'completed', 'Well-child visit - 5 year old', 'Vaccination due'),
(15, 15, 15, '2025-01-29 13:30:00', 'completed', 'High cholesterol follow-up', 'Started medication 3 months ago'),
(16, 16, 16, '2025-01-30 16:30:00', 'completed', 'Sore throat and ear pain', 'Symptoms started yesterday'),
(17, 17, 17, '2025-02-01 09:30:00', 'completed', 'Annual eye examination', 'Vision changes noted'),
(18, 18, 18, '2025-02-02 14:00:00', 'completed', 'Joint pain - arthritis consultation', 'Morning stiffness in hands'),
(19, 19, 19, '2025-02-03 11:00:00', 'completed', 'Sleep apnea evaluation', 'Partner reports loud snoring'),
(20, 20, 20, '2025-02-04 15:30:00', 'completed', 'Allergy testing consultation', 'Seasonal allergies worsening'),
(21, 21, 21, '2025-02-05 10:00:00', 'completed', 'Thyroid disorder follow-up', 'Fatigue and weight changes'),
(22, 22, 22, '2025-02-06 13:00:00', 'completed', 'Foot pain assessment', 'Pain in heel when walking'),
(23, 23, 23, '2025-02-07 16:00:00', 'completed', 'Pre-operative consultation', 'Scheduled for minor surgery'),
(24, 24, 24, '2025-02-08 09:00:00', 'completed', 'Vaccination - flu shot', 'Annual flu vaccine'),
(25, 25, 25, '2025-02-09 14:30:00', 'completed', 'Urinary tract infection symptoms', 'Painful urination for 2 days'),
(26, 26, 26, '2025-02-10 11:30:00', 'scheduled', 'Prenatal checkup - 32 weeks', 'Third trimester monitoring'),
(27, 27, 27, '2025-02-11 15:00:00', 'scheduled', 'Post-surgery follow-up', 'Check surgical site healing'),
(28, 28, 28, '2025-02-12 10:30:00', 'scheduled', 'Mental health counseling', 'Stress management needed'),
(29, 29, 29, '2025-02-13 13:30:00', 'cancelled', 'Routine dental referral consult', 'Patient cancelled - rescheduling'),
(30, 30, 30, '2025-02-14 16:30:00', 'scheduled', 'Annual wellness visit', 'Preventive care examination');

-- MEDICAL_RECORD TABLE

INSERT INTO Medical_Record (record_id, appointment_id, visit_date, diagnosis, visit_notes, treatment_plan, created_date, last_modified_date, lab_needed) VALUES
(1, 1, '2025-01-15', 'Healthy - no acute conditions', 'BP: 120/80, HR: 72, Temp: 98.6F. All systems normal.', 'Continue healthy lifestyle. Return in 1 year for next physical.', '2025-01-15 09:30:00', '2025-01-15 09:30:00', 'YES'),
(2, 2, '2025-01-16', 'Essential hypertension - controlled', 'BP: 128/82, improved from 145/95 last month. Patient compliant with medication.', 'Continue Lisinopril 10mg daily. Follow up in 3 months.', '2025-01-16 11:00:00', '2025-01-16 11:00:00', 'NO'),
(3, 3, '2025-01-17', 'Acute bronchitis', 'Productive cough, low-grade fever (100.2F), lungs with rhonchi bilaterally.', 'Prescribe antibiotic, cough suppressant. Rest and fluids. Return if worsening.', '2025-01-17 14:30:00', '2025-01-17 14:30:00', 'NO'),
(4, 4, '2025-01-18', 'Type 2 diabetes mellitus - well controlled', 'HbA1c 6.8%, down from 7.4%. Patient monitoring glucose regularly.', 'Continue Metformin 1000mg BID. Maintain diet and exercise. Recheck labs in 3 months.', '2025-01-18 11:30:00', '2025-01-18 11:30:00', 'YES'),
(5, 5, '2025-01-19', 'Medial meniscus tear - left knee', 'Swelling and tenderness over medial joint line. Positive McMurray test.', 'MRI ordered. RICE protocol. Refer to orthopedics. NSAIDs for pain.', '2025-01-19 16:00:00', '2025-01-19 16:00:00', 'NO'),
(6, 6, '2025-01-20', 'Pregnancy - 20 weeks, progressing normally', 'Fetal heart rate 145 bpm, fundal height appropriate. No complaints.', 'Continue prenatal vitamins. Anatomy ultrasound scheduled. Return in 4 weeks.', '2025-01-20 10:00:00', '2025-01-20 10:00:00', 'YES'),
(7, 7, '2025-01-21', 'Contact dermatitis', 'Erythematous rash with vesicles on bilateral forearms. History suggests allergic reaction.', 'Topical corticosteroid cream. Avoid suspected allergen. Follow up in 2 weeks.', '2025-01-21 13:30:00', '2025-01-21 13:30:00', 'NO'),
(8, 8, '2025-01-22', 'Chronic migraine without aura', 'Recurring headaches 15+ days/month. Photophobia and nausea present.', 'Start prophylactic medication. Keep headache diary. Avoid triggers. Follow up in 6 weeks.', '2025-01-22 16:30:00', '2025-01-22 16:30:00', 'NO'),
(9, 9, '2025-01-23', 'Non-cardiac chest pain - GERD likely', 'EKG normal, troponin negative. Pain associated with meals. No cardiac risk factors.', 'Trial of PPI therapy. Dietary modifications. Cardiology consult if symptoms persist.', '2025-01-23 10:30:00', '2025-01-23 10:30:00', 'YES'),
(10, 10, '2025-01-24', 'Major depressive disorder, moderate', 'PHQ-9 score: 16. Sleep disturbance, anhedonia, fatigue present for 6 weeks.', 'Start SSRI. Referral to therapist. Safety plan discussed. Follow up in 2 weeks.', '2025-01-24 15:00:00', '2025-01-24 15:00:00', 'NO'),
(11, 11, '2025-01-25', 'Acute lumbosacral strain', 'Tenderness over L4-L5, reduced flexion. No neurological deficits.', 'NSAIDs, muscle relaxants. Physical therapy referral. Avoid heavy lifting. Return in 2 weeks.', '2025-01-25 12:00:00', '2025-01-25 12:00:00', 'NO'),
(12, 12, '2025-01-26', 'Asthma - partially controlled', 'Using rescue inhaler 4-5 times/week. Nighttime symptoms 2x/week.', 'Step up controller therapy. Increase ICS dose. Asthma action plan reviewed. Follow up 1 month.', '2025-01-26 09:30:00', '2025-01-26 09:30:00', 'NO'),
(13, 13, '2025-01-27', 'Acute gastroenteritis', 'Nausea, vomiting, diarrhea x2 days. No fever. Mild dehydration.', 'Clear liquid diet, oral rehydration. Anti-emetic as needed. Return if severe symptoms.', '2025-01-27 15:30:00', '2025-01-27 15:30:00', 'NO'),
(14, 14, '2025-01-28', 'Well child - development on track', 'Height/weight 50th percentile. Vision/hearing normal. No concerns.', 'DTaP, IPV, MMR vaccines administered. Next visit age 6. Encourage outdoor play.', '2025-01-28 11:00:00', '2025-01-28 11:00:00', 'NO'),
(15, 15, '2025-01-29', 'Hyperlipidemia - improved', 'LDL decreased from 180 to 145 on statin therapy. HDL stable.', 'Continue Atorvastatin 20mg daily. Dietary counseling reinforced. Recheck lipids in 3 months.', '2025-01-29 14:00:00', '2025-01-29 14:00:00', 'YES'),
(16, 16, '2025-01-30', 'Acute pharyngitis with otitis media', 'Pharyngeal erythema, tonsillar exudate. Right TM red and bulging.', 'Antibiotic for 10 days. Pain management with acetaminophen. Follow up in 1 week.', '2025-01-30 17:00:00', '2025-01-30 17:00:00', 'YES'),
(17, 17, '2025-02-01', 'Presbyopia', 'Near vision decreased. Distance vision 20/20. Age-appropriate changes.', 'Reading glasses prescribed. Annual eye exams recommended. No other pathology noted.', '2025-02-01 10:00:00', '2025-02-01 10:00:00', 'NO'),
(18, 18, '2025-02-02', 'Rheumatoid arthritis - active', 'Symmetric polyarthritis, morning stiffness >1 hour. CRP elevated.', 'Start DMARD therapy. NSAID for symptom control. Rheumatology referral. Labs in 6 weeks.', '2025-02-02 14:30:00', '2025-02-02 14:30:00', 'YES'),
(19, 19, '2025-02-03', 'Obstructive sleep apnea - suspected', 'STOP-BANG score 6. Witnessed apneas, daytime somnolence, snoring.', 'Order sleep study. Weight loss counseling. Avoid alcohol before bed. Follow up with results.', '2025-02-03 11:30:00', '2025-02-03 11:30:00', 'NO'),
(20, 20, '2025-02-04', 'Allergic rhinitis - seasonal', 'Sneezing, rhinorrhea, itchy eyes during spring. History of seasonal allergies.', 'Intranasal corticosteroid spray. Antihistamine as needed. Environmental controls discussed.', '2025-02-04 16:00:00', '2025-02-04 16:00:00', 'YES'),
(21, 21, '2025-02-05', 'Hypothyroidism - undertreated', 'TSH 8.2 (elevated), fatigue, weight gain. On levothyroxine 75mcg.', 'Increase levothyroxine to 100mcg daily. Recheck TSH in 6 weeks. Monitor symptoms.', '2025-02-05 10:30:00', '2025-02-05 10:30:00', 'YES'),
(22, 22, '2025-02-06', 'Plantar fasciitis - right foot', 'Heel pain worse with first steps in morning. Tenderness at medial calcaneal tubercle.', 'Stretching exercises, arch supports. NSAIDs. Ice therapy. Physical therapy if not improved.', '2025-02-06 13:30:00', '2025-02-06 13:30:00', 'NO'),
(23, 23, '2025-02-07', 'Cholecystitis - chronic, pre-op clearance', 'Recurrent RUQ pain. Ultrasound shows gallstones. Cleared for laparoscopic cholecystectomy.', 'Surgery scheduled next week. Pre-op labs completed. Post-op instructions provided.', '2025-02-07 16:30:00', '2025-02-07 16:30:00', 'YES'),
(24, 24, '2025-02-08', 'Influenza prophylaxis', 'No contraindications to vaccination. No current illness.', 'Influenza vaccine administered. Monitor for adverse reactions. Return PRN.', '2025-02-08 09:30:00', '2025-02-08 09:30:00', 'NO'),
(25, 25, '2025-02-09', 'Acute cystitis', 'Dysuria, frequency, urgency. Urine dipstick positive for leukocytes and nitrites.', 'Antibiotic for 3 days. Increase fluid intake. Follow up if symptoms persist beyond 3 days.', '2025-02-09 15:00:00', '2025-02-09 15:00:00', 'YES'),
(26, 26, '2025-02-10', 'Pregnancy - 32 weeks, normal progression', 'BP 118/74, fundal height 32cm, fetal movement active. No edema.', 'Continue current care. Discuss birth plan. Weekly visits starting week 36.', '2025-02-10 12:00:00', '2025-02-10 12:00:00', 'YES'),
(27, 27, '2025-02-11', 'Post-operative - healing well', 'Surgical site clean, dry, intact. No signs of infection. Pain controlled.', 'Continue wound care. Remove sutures in 1 week. Activity restrictions for 2 more weeks.', '2025-02-11 15:30:00', '2025-02-11 15:30:00', 'NO'),
(28, 28, '2025-02-12', 'Adjustment disorder with anxiety', 'Stressors at work and home. GAD-7 score: 12. Coping poorly.', 'Counseling referral. Stress management techniques. Consider medication if no improvement. Follow up 4 weeks.', '2025-02-12 11:00:00', '2025-02-12 11:00:00', 'NO'),
(29, 29, '2025-02-13', 'Dental abscess - referred', 'Facial swelling, tooth pain. Requires urgent dental evaluation.', 'Emergency dental referral provided. Antibiotics started. Pain management.', '2025-02-13 14:00:00', '2025-02-13 14:00:00', 'NO'),
(30, 30, '2025-02-14', 'Annual wellness - age 30', 'Vital signs normal. No chronic conditions. Preventive care up to date.', 'Maintain healthy habits. Annual screening labs. Next visit in 1 year.', '2025-02-14 17:00:00', '2025-02-14 17:00:00', 'YES');

-- PRESCRIPTION TABLE

INSERT INTO Prescription (prescription_id, record_id, prescription_date, medication_name, dosage, frequency, instructions, duration) VALUES
(1, 2, '2025-01-16', 'Lisinopril', '10mg', 'Once daily', 'Take in the morning with or without food', '90 days'),
(2, 3, '2025-01-17', 'Azithromycin', '250mg', 'Once daily', 'Take on empty stomach, 1 hour before or 2 hours after meals', '5 days'),
(3, 3, '2025-01-17', 'Benzonatate', '100mg', 'Three times daily', 'Take with water, do not chew capsules', '10 days'),
(4, 4, '2025-01-18', 'Metformin', '1000mg', 'Twice daily', 'Take with meals to reduce stomach upset', '90 days'),
(5, 5, '2025-01-19', 'Ibuprofen', '600mg', 'Three times daily', 'Take with food or milk, maximum 3 doses per day', '14 days'),
(6, 6, '2025-01-20', 'Prenatal Vitamins', 'One tablet', 'Once daily', 'Take with food for better absorption', '30 days'),
(7, 7, '2025-01-21', 'Hydrocortisone Cream 1%', 'Apply thin layer', 'Twice daily', 'Apply to affected areas only, avoid face', '14 days'),
(8, 8, '2025-01-22', 'Topiramate', '25mg', 'Once daily at bedtime', 'Start low dose, may increase after 2 weeks. Drink plenty of water.', '30 days'),
(9, 9, '2025-01-23', 'Omeprazole', '20mg', 'Once daily', 'Take 30 minutes before breakfast', '30 days'),
(10, 10, '2025-01-24', 'Sertraline', '50mg', 'Once daily', 'Take in morning with food. May take 4-6 weeks for full effect.', '30 days'),
(11, 11, '2025-01-25', 'Cyclobenzaprine', '10mg', 'Three times daily', 'May cause drowsiness, do not drive if affected', '7 days'),
(12, 11, '2025-01-25', 'Naproxen', '500mg', 'Twice daily', 'Take with food or milk to prevent stomach upset', '14 days'),
(13, 12, '2025-01-26', 'Fluticasone Inhaler', '250mcg', 'Two puffs twice daily', 'Rinse mouth after use. Use daily even when feeling well.', '30 days'),
(14, 13, '2025-01-27', 'Ondansetron', '4mg', 'Every 8 hours as needed', 'Dissolve on tongue, can be taken with or without water', '5 days'),
(15, 15, '2025-01-29', 'Atorvastatin', '20mg', 'Once daily at bedtime', 'Take in evening for best effect. Avoid grapefruit juice.', '90 days'),
(16, 16, '2025-01-30', 'Amoxicillin', '500mg', 'Three times daily', 'Take every 8 hours, complete full course even if feeling better', '10 days'),
(17, 18, '2025-02-02', 'Methotrexate', '15mg', 'Once weekly', 'Take same day each week. Avoid alcohol. Use contraception.', '30 days'),
(18, 18, '2025-02-02', 'Folic Acid', '1mg', 'Once daily', 'Take on non-methotrexate days', '30 days'),
(19, 20, '2025-02-04', 'Fluticasone Nasal Spray', 'Two sprays each nostril', 'Once daily', 'Shake well before use, may take several days for full benefit', '30 days'),
(20, 20, '2025-02-04', 'Cetirizine', '10mg', 'Once daily', 'Can take with or without food, may cause mild drowsiness', '30 days'),
(21, 21, '2025-02-05', 'Levothyroxine', '100mcg', 'Once daily', 'Take on empty stomach 30-60 minutes before breakfast. Do not take with calcium or iron.', '90 days'),
(22, 23, '2025-02-07', 'Cephalexin', '500mg', 'Four times daily', 'Start after surgery as directed. Take every 6 hours.', '7 days'),
(23, 25, '2025-02-09', 'Nitrofurantoin', '100mg', 'Twice daily', 'Take with food or milk. Drink plenty of water.', '3 days'),
(24, 26, '2025-02-10', 'Iron Supplement', '325mg', 'Once daily', 'Take with vitamin C for better absorption. May cause constipation.', '30 days'),
(25, 27, '2025-02-11', 'Hydrocodone/Acetaminophen', '5mg/325mg', 'Every 6 hours as needed', 'For pain control. Do not exceed 6 tablets in 24 hours. May cause drowsiness.', '5 days'),
(26, 28, '2025-02-12', 'Buspirone', '10mg', 'Twice daily', 'Take consistently with or without food. May take 2-4 weeks for full effect.', '30 days'),
(27, 29, '2025-02-13', 'Clindamycin', '300mg', 'Four times daily', 'Take with full glass of water. Complete entire course.', '7 days'),
(28, 8, '2025-01-22', 'Sumatriptan', '50mg', 'At onset of migraine', 'May repeat once after 2 hours if needed. Maximum 2 doses per 24 hours.', '30 days'),
(29, 12, '2025-01-26', 'Albuterol Inhaler', '90mcg', 'Two puffs every 4-6 hours as needed', 'Use as rescue medication for acute symptoms', '30 days'),
(30, 22, '2025-02-06', 'Diclofenac Gel', 'Apply to affected area', 'Four times daily', 'Massage into skin over painful area. Wash hands after application.', '30 days');

-- BILLING TABLE

INSERT INTO Billing (billing_id, appointment_id, billing_date, total_amount, amount_paid, amount_due, payment_status) VALUES
(1, 1, '2025-01-15 10:00:00', 250.00, 250.00, 0.00, 'paid'),
(2, 2, '2025-01-16 11:30:00', 150.00, 150.00, 0.00, 'paid'),
(3, 3, '2025-01-17 15:00:00', 200.00, 50.00, 150.00, 'partial'),
(4, 4, '2025-01-18 12:00:00', 325.00, 325.00, 0.00, 'paid'),
(5, 5, '2025-01-19 16:30:00', 275.00, 0.00, 275.00, 'pending'),
(6, 6, '2025-01-20 10:30:00', 180.00, 180.00, 0.00, 'paid'),
(7, 7, '2025-01-21 14:00:00', 165.00, 165.00, 0.00, 'paid'),
(8, 8, '2025-01-22 17:00:00', 220.00, 100.00, 120.00, 'partial'),
(9, 9, '2025-01-23 11:00:00', 450.00, 450.00, 0.00, 'paid'),
(10, 10, '2025-01-24 15:30:00', 200.00, 200.00, 0.00, 'paid'),
(11, 11, '2025-01-25 12:30:00', 185.00, 0.00, 185.00, 'pending'),
(12, 12, '2025-01-26 10:00:00', 175.00, 175.00, 0.00, 'paid'),
(13, 13, '2025-01-27 16:00:00', 160.00, 160.00, 0.00, 'paid'),
(14, 14, '2025-01-28 11:30:00', 225.00, 225.00, 0.00, 'paid'),
(15, 15, '2025-01-29 14:30:00', 295.00, 295.00, 0.00, 'paid'),
(16, 16, '2025-01-30 17:30:00', 245.00, 50.00, 195.00, 'overdue'),
(17, 17, '2025-02-01 10:30:00', 300.00, 300.00, 0.00, 'paid'),
(18, 18, '2025-02-02 15:00:00', 380.00, 0.00, 380.00, 'pending'),
(19, 19, '2025-02-03 12:00:00', 190.00, 190.00, 0.00, 'paid'),
(20, 20, '2025-02-04 16:30:00', 335.00, 335.00, 0.00, 'paid'),
(21, 21, '2025-02-05 11:00:00', 265.00, 100.00, 165.00, 'partial'),
(22, 22, '2025-02-06 14:00:00', 155.00, 155.00, 0.00, 'paid'),
(23, 23, '2025-02-07 17:00:00', 425.00, 425.00, 0.00, 'paid'),
(24, 24, '2025-02-08 10:00:00', 85.00, 85.00, 0.00, 'paid'),
(25, 25, '2025-02-09 15:30:00', 215.00, 0.00, 215.00, 'pending'),
(26, 26, '2025-02-10 12:30:00', 195.00, 0.00, 195.00, 'pending'),
(27, 27, '2025-02-11 16:00:00', 140.00, 0.00, 140.00, 'pending'),
(28, 28, '2025-02-12 11:30:00', 210.00, 0.00, 210.00, 'pending'),
(29, 29, '2025-02-13 14:30:00', 175.00, 0.00, 175.00, 'pending'),
(30, 30, '2025-02-14 17:30:00', 230.00, 0.00, 230.00, 'pending');

-- BILLING_ITEM TABLE
INSERT INTO Billing_Item (item_id, billing_id, service_type, description, unit_cost, quantity, total_cost) VALUES
(1, 1, 'consultation', 'Annual Physical Examination', 150.00, 1, 150.00),
(2, 1, 'lab', 'Complete Blood Count', 50.00, 1, 50.00),
(3, 1, 'lab', 'Lipid Panel', 50.00, 1, 50.00),
(4, 2, 'consultation', 'Follow-up Visit - Hypertension', 150.00, 1, 150.00),
(5, 3, 'consultation', 'Sick Visit - Respiratory', 150.00, 1, 150.00),
(6, 3, 'medication', 'Antibiotic Prescription', 50.00, 1, 50.00),
(7, 4, 'consultation', 'Diabetes Management Consultation', 175.00, 1, 175.00),
(8, 4, 'lab', 'HbA1c Test', 75.00, 1, 75.00),
(9, 4, 'lab', 'Glucose Fasting', 75.00, 1, 75.00),
(10, 5, 'consultation', 'Sports Medicine Consultation', 200.00, 1, 200.00),
(11, 5, 'procedure', 'Knee X-Ray (2 views)', 75.00, 1, 75.00),
(12, 6, 'consultation', 'Prenatal Visit - Second Trimester', 180.00, 1, 180.00),
(13, 7, 'consultation', 'Dermatology Consultation', 165.00, 1, 165.00),
(14, 8, 'consultation', 'Neurology Consultation - Migraine', 220.00, 1, 220.00),
(15, 9, 'consultation', 'Emergency Consultation - Chest Pain', 250.00, 1, 250.00),
(16, 9, 'lab', 'ECG/EKG', 100.00, 1, 100.00),
(17, 9, 'lab', 'Cardiac Enzyme Panel', 100.00, 1, 100.00),
(18, 10, 'consultation', 'Mental Health Evaluation', 200.00, 1, 200.00),
(19, 11, 'consultation', 'Orthopedic Consultation', 185.00, 1, 185.00),
(20, 12, 'consultation', 'Pulmonology Follow-up', 175.00, 1, 175.00),
(21, 13, 'consultation', 'Gastroenterology Consultation', 160.00, 1, 160.00),
(22, 14, 'consultation', 'Well Child Visit', 125.00, 1, 125.00),
(23, 14, 'medication', 'Vaccination - DTaP', 50.00, 1, 50.00),
(24, 14, 'medication', 'Vaccination - MMR', 50.00, 1, 50.00),
(25, 15, 'consultation', 'Cardiology Follow-up', 175.00, 1, 175.00),
(26, 15, 'lab', 'Lipid Panel Comprehensive', 120.00, 1, 120.00),
(27, 16, 'consultation', 'ENT Consultation', 170.00, 1, 170.00),
(28, 16, 'lab', 'Rapid Strep Test', 40.00, 1, 40.00),
(29, 16, 'medication', 'Antibiotic Prescription', 35.00, 1, 35.00),
(30, 17, 'consultation', 'Ophthalmology Examination', 200.00, 1, 200.00),
(31, 17, 'procedure', 'Visual Acuity Test', 50.00, 1, 50.00),
(32, 17, 'procedure', 'Tonometry (Glaucoma Screening)', 50.00, 1, 50.00),
(33, 18, 'consultation', 'Rheumatology Consultation', 225.00, 1, 225.00),
(34, 18, 'lab', 'Rheumatoid Factor', 80.00, 1, 80.00),
(35, 18, 'lab', 'CRP Test', 75.00, 1, 75.00),
(36, 19, 'consultation', 'Sleep Medicine Consultation', 190.00, 1, 190.00),
(37, 20, 'consultation', 'Allergy Consultation', 185.00, 1, 185.00),
(38, 20, 'lab', 'Allergy Panel - Environmental', 150.00, 1, 150.00),
(39, 21, 'consultation', 'Endocrinology Consultation', 190.00, 1, 190.00),
(40, 21, 'lab', 'Thyroid Function Panel', 75.00, 1, 75.00),
(41, 22, 'consultation', 'Podiatry Consultation', 155.00, 1, 155.00),
(42, 23, 'consultation', 'Pre-operative Assessment', 200.00, 1, 200.00),
(43, 23, 'lab', 'Pre-operative Lab Panel', 125.00, 1, 125.00),
(44, 23, 'procedure', 'EKG Pre-operative', 100.00, 1, 100.00),
(45, 24, 'medication', 'Influenza Vaccine', 45.00, 1, 45.00),
(46, 24, 'consultation', 'Vaccination Administration', 40.00, 1, 40.00),
(47, 25, 'consultation', 'Urology Consultation', 165.00, 1, 165.00),
(48, 25, 'lab', 'Urinalysis with Culture', 50.00, 1, 50.00),
(49, 26, 'consultation', 'Prenatal Visit - Third Trimester', 145.00, 1, 145.00),
(50, 26, 'lab', 'Group B Strep Culture', 50.00, 1, 50.00);

-- INSURANCE_POLICY TABLE

INSERT INTO Insurance_Policy (policy_id, provider_name, policy_number, coverage_type, coverage_start_date, coverage_end_date, coverage_amount) VALUES
(1, 'Blue Cross Blue Shield', 'BCBS-2024-001234', 'PPO Premium', '2024-01-01', '2024-12-31', 500000.00),
(2, 'United Healthcare', 'UHC-2024-005678', 'HMO Standard', '2024-01-01', '2024-12-31', 300000.00),
(3, 'Aetna', 'AET-2024-009012', 'PPO Gold', '2024-01-01', '2024-12-31', 750000.00),
(4, 'Cigna', 'CIG-2024-003456', 'EPO Plus', '2024-02-01', '2025-01-31', 400000.00),
(5, 'Humana', 'HUM-2024-007890', 'PPO Basic', '2024-01-01', '2024-12-31', 250000.00),
(6, 'Kaiser Permanente', 'KP-2024-001111', 'HMO Platinum', '2024-03-01', '2025-02-28', 1000000.00),
(7, 'Blue Cross Blue Shield', 'BCBS-2024-002222', 'PPO Standard', '2024-01-01', '2024-12-31', 350000.00),
(8, 'United Healthcare', 'UHC-2024-003333', 'PPO Premium', '2024-04-01', '2025-03-31', 600000.00),
(9, 'Aetna', 'AET-2024-004444', 'HMO Basic', '2024-01-01', '2024-12-31', 200000.00),
(10, 'Cigna', 'CIG-2024-005555', 'PPO Gold', '2024-05-01', '2025-04-30', 550000.00),
(11, 'Humana', 'HUM-2024-006666', 'EPO Standard', '2024-01-01', '2024-12-31', 325000.00),
(12, 'Kaiser Permanente', 'KP-2024-007777', 'HMO Gold', '2024-06-01', '2025-05-31', 450000.00),
(13, 'Blue Cross Blue Shield', 'BCBS-2024-008888', 'EPO Plus', '2024-01-01', '2024-12-31', 475000.00),
(14, 'United Healthcare', 'UHC-2024-009999', 'HMO Premium', '2024-07-01', '2025-06-30', 700000.00),
(15, 'Aetna', 'AET-2024-010101', 'PPO Platinum', '2024-01-01', '2024-12-31', 850000.00),
(16, 'Cigna', 'CIG-2024-011111', 'HMO Standard', '2024-08-01', '2025-07-31', 280000.00),
(17, 'Humana', 'HUM-2024-012121', 'PPO Premium', '2024-01-01', '2024-12-31', 525000.00),
(18, 'Kaiser Permanente', 'KP-2024-013131', 'EPO Gold', '2024-09-01', '2025-08-31', 625000.00),
(19, 'Blue Cross Blue Shield', 'BCBS-2024-014141', 'HMO Plus', '2024-01-01', '2024-12-31', 375000.00),
(20, 'United Healthcare', 'UHC-2024-015151', 'PPO Gold', '2024-10-01', '2025-09-30', 575000.00),
(21, 'Aetna', 'AET-2024-016161', 'EPO Standard', '2024-01-01', '2024-12-31', 300000.00),
(22, 'Cigna', 'CIG-2024-017171', 'PPO Platinum', '2024-11-01', '2025-10-31', 900000.00),
(23, 'Humana', 'HUM-2024-018181', 'HMO Gold', '2024-01-01', '2024-12-31', 425000.00),
(24, 'Kaiser Permanente', 'KP-2024-019191', 'PPO Standard', '2024-12-01', '2025-11-30', 380000.00),
(25, 'Blue Cross Blue Shield', 'BCBS-2024-020201', 'EPO Premium', '2024-01-01', '2024-12-31', 650000.00),
(26, 'United Healthcare', 'UHC-2025-021211', 'HMO Platinum', '2025-01-01', '2025-12-31', 950000.00),
(27, 'Aetna', 'AET-2025-022221', 'PPO Plus', '2025-01-01', '2025-12-31', 500000.00),
(28, 'Cigna', 'CIG-2025-023231', 'HMO Gold', '2025-02-01', '2026-01-31', 475000.00),
(29, 'Humana', 'HUM-2025-024241', 'EPO Platinum', '2025-01-01', '2025-12-31', 825000.00),
(30, 'Kaiser Permanente', 'KP-2025-025251', 'PPO Premium', '2025-03-01', '2026-02-28', 725000.00);

-- PATIENT_INSURANCE TABLE

INSERT INTO Patient_Insurance (patient_id, policy_id, relationship_type, effective_date) VALUES
(1, 1, 'Self', '2024-01-01'),
(2, 2, 'Self', '2024-01-01'),
(3, 3, 'Self', '2024-01-01'),
(4, 4, 'Self', '2024-02-01'),
(5, 5, 'Self', '2024-01-01'),
(6, 6, 'Self', '2024-03-01'),
(7, 7, 'Self', '2024-01-01'),
(8, 8, 'Self', '2024-04-01'),
(9, 9, 'Self', '2024-01-01'),
(10, 10, 'Self', '2024-05-01'),
(11, 11, 'Self', '2024-01-01'),
(12, 12, 'Self', '2024-06-01'),
(13, 13, 'Self', '2024-01-01'),
(14, 14, 'Spouse', '2024-07-01'),
(15, 15, 'Self', '2024-01-01'),
(16, 16, 'Self', '2024-08-01'),
(17, 17, 'Self', '2024-01-01'),
(18, 18, 'Self', '2024-09-01'),
(19, 19, 'Self', '2024-01-01'),
(20, 20, 'Self', '2024-10-01'),
(21, 21, 'Self', '2024-01-01'),
(22, 22, 'Self', '2024-11-01'),
(23, 23, 'Self', '2024-01-01'),
(24, 24, 'Dependent', '2024-12-01'),
(25, 25, 'Self', '2024-01-01'),
(26, 26, 'Self', '2025-01-01'),
(27, 27, 'Self', '2025-01-01'),
(28, 28, 'Self', '2025-02-01'),
(29, 29, 'Spouse', '2025-01-01'),
(30, 30, 'Self', '2025-03-01');

-- INSURANCE_CLAIM TABLE

INSERT INTO Insurance_Claim (claim_id, policy_id, billing_id, claim_date, claim_amount, approval_date, notes, status) VALUES
(1, 1, 1, '2025-01-16 09:00:00', 200.00, '2025-01-20 14:30:00', 'Annual physical - approved with standard coverage', 'approved'),
(2, 2, 2, '2025-01-17 10:00:00', 120.00, '2025-01-21 11:00:00', 'Follow-up visit approved', 'approved'),
(3, 3, 3, '2025-01-18 11:00:00', 150.00, '2025-01-25 15:30:00', 'Sick visit claim approved', 'approved'),
(4, 4, 4, '2025-01-19 09:30:00', 260.00, '2025-01-23 10:00:00', 'Diabetes management approved', 'approved'),
(5, 5, 5, '2025-01-20 10:00:00', 220.00, '2025-02-01 09:00:00', 'Sports injury consultation pending review', 'under_review'),
(6, 6, 6, '2025-01-21 11:00:00', 144.00, '2025-01-26 13:00:00', 'Prenatal care approved - 80% coverage', 'approved'),
(7, 7, 7, '2025-01-22 12:00:00', 132.00, '2025-01-28 14:00:00', 'Dermatology consultation approved', 'approved'),
(8, 8, 8, '2025-01-23 13:00:00', 176.00, '2025-01-30 16:00:00', 'Neurology consultation approved', 'approved'),
(9, 9, 9, '2025-01-24 10:30:00', 360.00, '2025-01-27 11:30:00', 'Emergency consultation approved', 'approved'),
(10, 10, 10, '2025-01-25 14:00:00', 160.00, '2025-02-02 10:00:00', 'Mental health evaluation approved', 'approved'),
(11, 11, 11, '2025-01-26 11:00:00', 148.00, '2025-02-05 12:00:00', 'Under review for orthopedic consultation', 'under_review'),
(12, 12, 12, '2025-01-27 09:30:00', 140.00, '2025-02-01 15:00:00', 'Pulmonology follow-up approved', 'approved'),
(13, 13, 13, '2025-01-28 12:00:00', 128.00, '2025-02-03 09:30:00', 'GI consultation approved', 'approved'),
(14, 14, 14, '2025-01-29 10:00:00', 180.00, '2025-02-04 11:00:00', 'Well child visit approved', 'approved'),
(15, 15, 15, '2025-01-30 13:00:00', 236.00, '2025-02-06 14:00:00', 'Cardiology follow-up approved', 'approved'),
(16, 16, 16, '2025-01-31 11:30:00', 196.00, '2025-02-15 10:00:00', 'Payment overdue - claim approved but patient balance remains', 'approved'),
(17, 17, 17, '2025-02-01 15:00:00', 240.00, '2025-02-08 13:00:00', 'Ophthalmology exam approved', 'approved'),
(18, 18, 18, '2025-02-02 16:00:00', 304.00, '2025-02-18 09:00:00', 'Rheumatology claim submitted', 'submitted'),
(19, 19, 19, '2025-02-03 13:00:00', 152.00, '2025-02-10 11:30:00', 'Sleep medicine consultation approved', 'approved'),
(20, 20, 20, '2025-02-04 17:00:00', 268.00, '2025-02-12 14:00:00', 'Allergy consultation and testing approved', 'approved'),
(21, 21, 21, '2025-02-05 12:00:00', 212.00, '2025-02-20 10:00:00', 'Endocrinology claim under review', 'under_review'),
(22, 22, 22, '2025-02-06 14:30:00', 124.00, '2025-02-13 15:00:00', 'Podiatry consultation approved', 'approved'),
(23, 23, 23, '2025-02-07 17:30:00', 340.00, '2025-02-14 11:00:00', 'Pre-operative assessment approved', 'approved'),
(24, 24, 24, '2025-02-08 10:30:00', 68.00, '2025-02-15 13:00:00', 'Vaccination approved', 'approved'),
(25, 25, 25, '2025-02-09 16:00:00', 172.00, '2025-02-25 09:00:00', 'Claim submitted for review', 'submitted'),
(26, 26, 26, '2025-02-10 13:00:00', 156.00, '2025-02-28 10:00:00', 'Draft - preparing claim documentation', 'draft'),
(27, 27, 27, '2025-02-11 16:30:00', 112.00, '2025-03-01 11:00:00', 'Draft - post-operative follow-up', 'draft'),
(28, 28, 28, '2025-02-12 12:00:00', 168.00, '2025-03-05 14:00:00', 'Draft - mental health claim preparation', 'draft'),
(29, 29, 29, '2025-02-13 15:00:00', 140.00, '2025-02-20 09:00:00', 'Dental referral - denied as not covered', 'denied'),
(30, 30, 30, '2025-02-14 18:00:00', 184.00, '2025-03-10 10:00:00', 'Draft - annual wellness claim', 'draft');

-- LAB_TEST TABLE

INSERT INTO Lab_Test (test_id, test_name, test_description, standard_cost) VALUES
(1, 'Complete Blood Count (CBC)', 'Measures red cells, white cells, platelets, hemoglobin, and hematocrit', 45.00),
(2, 'Basic Metabolic Panel (BMP)', 'Tests glucose, calcium, electrolytes, and kidney function', 55.00),
(3, 'Comprehensive Metabolic Panel (CMP)', 'Extended panel including liver function tests', 75.00),
(4, 'Lipid Panel', 'Cholesterol, HDL, LDL, and triglycerides measurement', 60.00),
(5, 'Hemoglobin A1C (HbA1c)', 'Average blood sugar over past 2-3 months', 65.00),
(6, 'Thyroid Stimulating Hormone (TSH)', 'Thyroid function screening test', 50.00),
(7, 'Free T4 (Thyroxine)', 'Thyroid hormone level measurement', 55.00),
(8, 'Vitamin D 25-Hydroxy', 'Vitamin D level assessment', 85.00),
(9, 'Vitamin B12', 'B12 deficiency screening', 70.00),
(10, 'Urinalysis', 'Urine chemical and microscopic analysis', 25.00),
(11, 'Urine Culture', 'Bacterial identification in urine', 65.00),
(12, 'Pregnancy Test (hCG)', 'Qualitative or quantitative pregnancy hormone', 40.00),
(13, 'Prostate Specific Antigen (PSA)', 'Prostate cancer screening marker', 75.00),
(14, 'C-Reactive Protein (CRP)', 'Inflammation marker', 60.00),
(15, 'Erythrocyte Sedimentation Rate (ESR)', 'Non-specific inflammation indicator', 35.00),
(16, 'Rheumatoid Factor', 'Autoimmune arthritis screening', 70.00),
(17, 'Antinuclear Antibody (ANA)', 'Autoimmune disorder screening', 90.00),
(18, 'Liver Function Tests (LFT)', 'AST, ALT, alkaline phosphatase, bilirubin', 80.00),
(19, 'Prothrombin Time (PT/INR)', 'Blood clotting measurement', 45.00),
(20, 'Partial Thromboplastin Time (PTT)', 'Clotting pathway assessment', 50.00),
(21, 'Iron Studies Panel', 'Iron, TIBC, ferritin, transferrin saturation', 95.00),
(22, 'Magnesium Level', 'Serum magnesium measurement', 40.00),
(23, 'Troponin I', 'Cardiac muscle damage marker', 120.00),
(24, 'Brain Natriuretic Peptide (BNP)', 'Heart failure marker', 135.00),
(25, 'Strep Throat Rapid Test', 'Group A Streptococcus detection', 35.00),
(26, 'Influenza Rapid Test', 'Flu A and B detection', 55.00),
(27, 'COVID-19 PCR Test', 'SARS-CoV-2 detection', 100.00),
(28, 'Stool Occult Blood Test', 'Hidden blood in stool detection', 30.00),
(29, 'Helicobacter Pylori Test', 'H. pylori infection detection', 85.00),
(30, 'Celiac Disease Antibody Panel', 'tTG-IgA and total IgA', 145.00);


-- LAB_REPORT TABLE

INSERT INTO Lab_Report (report_id, record_id, user_id, test_id, test_date, result, status, report_file_path, notes) VALUES
(1, 1, 61, 1, '2025-01-15', 'WBC: 7.2, RBC: 4.8, Hgb: 14.2, Hct: 42%, Platelets: 245 - All values within normal range', 'completed', '/lab_reports/2025/01/report_001_cbc.pdf', 'Normal complete blood count'),
(2, 1, 61, 4, '2025-01-15', 'Total Cholesterol: 185, HDL: 55, LDL: 105, Triglycerides: 125 - Optimal levels', 'reviewed', '/lab_reports/2025/01/report_002_lipid.pdf', 'Excellent lipid profile'),
(3, 4, 62, 5, '2025-01-18', 'HbA1c: 6.8% - Good diabetic control', 'completed', '/lab_reports/2025/01/report_003_hba1c.pdf', 'Improved from previous 7.4%'),
(4, 4, 62, 2, '2025-01-18', 'Glucose: 118, BUN: 16, Creatinine: 0.9, eGFR: >90 - Normal kidney function', 'reviewed', '/lab_reports/2025/01/report_004_bmp.pdf', 'Kidney function excellent'),
(5, 6, 63, 12, '2025-01-20', 'hCG: 45,200 mIU/mL - Consistent with 20 weeks gestation', 'completed', '/lab_reports/2025/01/report_005_hcg.pdf', 'Pregnancy progressing normally'),
(6, 6, 63, 1, '2025-01-20', 'Hgb: 11.8, Hct: 35% - Mild anemia of pregnancy', 'reviewed', '/lab_reports/2025/01/report_006_cbc.pdf', 'Iron supplementation recommended'),
(7, 9, 64, 23, '2025-01-23', 'Troponin I: <0.01 ng/mL - Negative for cardiac injury', 'completed', '/lab_reports/2025/01/report_007_troponin.pdf', 'Rules out myocardial infarction'),
(8, 9, 64, 1, '2025-01-23', 'All blood counts within normal limits', 'reviewed', '/lab_reports/2025/01/report_008_cbc.pdf', 'No abnormalities detected'),
(9, 15, 65, 4, '2025-01-29', 'Total Cholesterol: 225, LDL: 145, HDL: 48, Triglycerides: 160 - Improved on statin', 'completed', '/lab_reports/2025/01/report_009_lipid.pdf', 'LDL decreased from 180'),
(10, 15, 65, 18, '2025-01-29', 'AST: 28, ALT: 32, Alk Phos: 78 - Liver function normal on statin', 'reviewed', '/lab_reports/2025/01/report_010_lft.pdf', 'Safe to continue medication'),
(11, 16, 66, 25, '2025-01-30', 'Positive for Group A Streptococcus', 'completed', '/lab_reports/2025/01/report_011_strep.pdf', 'Strep throat confirmed'),
(12, 16, 66, 1, '2025-01-30', 'WBC: 12.5 - Elevated, consistent with bacterial infection', 'reviewed', '/lab_reports/2025/01/report_012_cbc.pdf', 'Leukocytosis due to infection'),
(13, 18, 67, 16, '2025-02-02', 'Rheumatoid Factor: 85 IU/mL (elevated) - Positive', 'completed', '/lab_reports/2025/02/report_013_rf.pdf', 'Consistent with RA diagnosis'),
(14, 18, 67, 14, '2025-02-02', 'CRP: 18 mg/L - Significantly elevated', 'reviewed', '/lab_reports/2025/02/report_014_crp.pdf', 'Active inflammation present'),
(15, 18, 67, 1, '2025-02-02', 'Mild normocytic anemia, WBC normal', 'reviewed', '/lab_reports/2025/02/report_015_cbc.pdf', 'Anemia of chronic disease'),
(16, 20, 68, 17, '2025-02-04', 'ANA: Positive 1:160, speckled pattern', 'completed', '/lab_reports/2025/02/report_016_ana.pdf', 'Further autoimmune workup needed'),
(17, 20, 68, 1, '2025-02-04', 'Eosinophils: 8% (elevated) - Allergic response', 'reviewed', '/lab_reports/2025/02/report_017_cbc.pdf', 'Eosinophilia consistent with allergies'),
(18, 21, 69, 6, '2025-02-05', 'TSH: 8.2 mIU/L - Elevated (hypothyroid)', 'completed', '/lab_reports/2025/02/report_018_tsh.pdf', 'Dose adjustment needed'),
(19, 21, 69, 7, '2025-02-05', 'Free T4: 0.7 ng/dL - Low', 'reviewed', '/lab_reports/2025/02/report_019_t4.pdf', 'Confirms hypothyroidism'),
(20, 23, 70, 3, '2025-02-07', 'All values within normal limits - cleared for surgery', 'completed', '/lab_reports/2025/02/report_020_cmp.pdf', 'Pre-operative clearance granted'),
(21, 23, 70, 1, '2025-02-07', 'Hemoglobin adequate for surgery: 13.8 g/dL', 'reviewed', '/lab_reports/2025/02/report_021_cbc.pdf', 'No transfusion anticipated'),
(22, 25, 71, 10, '2025-02-09', 'Leukocyte esterase: positive, Nitrites: positive, WBC: many', 'completed', '/lab_reports/2025/02/report_022_ua.pdf', 'Urinary tract infection confirmed'),
(23, 25, 71, 11, '2025-02-09', 'E. coli >100,000 CFU/mL, sensitive to nitrofurantoin', 'pending', '/lab_reports/2025/02/report_023_urine_culture.pdf', 'Culture in progress, preliminary results'),
(24, 26, 72, 12, '2025-02-10', 'hCG: 87,500 mIU/mL - Normal for 32 weeks', 'completed', '/lab_reports/2025/02/report_024_hcg.pdf', 'Pregnancy hormone levels appropriate'),
(25, 26, 72, 1, '2025-02-10', 'Hemoglobin: 11.2 g/dL - Mild anemia', 'reviewed', '/lab_reports/2025/02/report_025_cbc.pdf', 'Continue iron supplementation'),
(26, 30, 73, 4, '2025-02-14', 'Total Cholesterol: 172, HDL: 62, LDL: 95, Triglycerides: 75', 'pending', '/lab_reports/2025/02/report_026_lipid.pdf', 'Pending physician review'),
(27, 30, 73, 6, '2025-02-14', 'TSH: 2.1 mIU/L - Normal thyroid function', 'pending', '/lab_reports/2025/02/report_027_tsh.pdf', 'Results just received'),
(28, 1, 74, 8, '2025-01-15', 'Vitamin D: 28 ng/mL - Insufficient', 'archived', '/lab_reports/2025/01/report_028_vitd.pdf', 'Supplementation started'),
(29, 4, 75, 10, '2025-01-18', 'Glucose: trace, Protein: negative, Ketones: negative', 'completed', '/lab_reports/2025/01/report_029_ua.pdf', 'Diabetic monitoring - no protein spillage'),
(30, 15, 76, 18, '2025-01-29', 'AST: 24, ALT: 28, Total Bilirubin: 0.6 - All normal', 'reviewed', '/lab_reports/2025/01/report_030_lft.pdf', 'Liver tolerating statin well');

-- SPECIALIZATION TABLE

INSERT INTO Specialization (specialization_id, specialization_name, description) VALUES
(1, 'Internal Medicine', 'Diagnosis and treatment of adult diseases, preventive care, and chronic disease management'),
(2, 'Pediatrics', 'Medical care for infants, children, and adolescents including growth and development'),
(3, 'Cardiology', 'Heart and cardiovascular system disorders, including hypertension and heart disease'),
(4, 'Dermatology', 'Skin, hair, and nail conditions including acne, eczema, and skin cancer'),
(5, 'Orthopedics', 'Musculoskeletal system including bones, joints, ligaments, tendons, and muscles'),
(6, 'Obstetrics and Gynecology', 'Women''s reproductive health, pregnancy, and childbirth'),
(7, 'Neurology', 'Nervous system disorders including headaches, epilepsy, and stroke'),
(8, 'Psychiatry', 'Mental health disorders, emotional problems, and behavioral issues'),
(9, 'Gastroenterology', 'Digestive system disorders including stomach, intestines, liver, and pancreas'),
(10, 'Pulmonology', 'Respiratory system and lung diseases including asthma and COPD'),
(11, 'Endocrinology', 'Hormone-related disorders including diabetes, thyroid, and metabolic diseases'),
(12, 'Rheumatology', 'Autoimmune and inflammatory joint and connective tissue diseases'),
(13, 'Nephrology', 'Kidney diseases and disorders, dialysis management'),
(14, 'Urology', 'Urinary tract and male reproductive system disorders'),
(15, 'Ophthalmology', 'Eye diseases, vision problems, and eye surgery'),
(16, 'Otolaryngology (ENT)', 'Ear, nose, throat, head, and neck disorders'),
(17, 'Allergy and Immunology', 'Allergic reactions, asthma, and immune system disorders'),
(18, 'Oncology', 'Cancer diagnosis, treatment, and management'),
(19, 'Hematology', 'Blood disorders including anemia, clotting disorders, and blood cancers'),
(20, 'Infectious Disease', 'Diagnosis and treatment of infections caused by bacteria, viruses, parasites, and fungi'),
(21, 'Family Medicine', 'Comprehensive healthcare for all ages with continuity of care'),
(22, 'Emergency Medicine', 'Acute illness and injury care in emergency settings'),
(23, 'Anesthesiology', 'Pain management and anesthesia for surgical procedures'),
(24, 'Radiology', 'Medical imaging including X-rays, CT, MRI, and ultrasound interpretation'),
(25, 'Pathology', 'Laboratory medicine and disease diagnosis through tissue and fluid analysis'),
(26, 'Physical Medicine and Rehabilitation', 'Restoration of function and quality of life after injury or illness'),
(27, 'Sports Medicine', 'Athletic injuries, performance optimization, and exercise-related conditions'),
(28, 'Geriatrics', 'Healthcare for elderly patients and age-related conditions'),
(29, 'Sleep Medicine', 'Sleep disorders including sleep apnea, insomnia, and narcolepsy'),
(30, 'Pain Management', 'Chronic and acute pain evaluation and treatment');


-- DOCTOR_SPECIALIZATION TABLE

INSERT INTO Doctor_Specialization (doctor_id, specialization_id, assigned_date) VALUES
(1, 1, '2015-06-01'),
(2, 2, '2016-03-15'),
(3, 3, '2014-09-10'),
(4, 4, '2017-01-20'),
(5, 5, '2015-11-08'),
(6, 6, '2018-04-12'),
(7, 7, '2016-07-25'),
(8, 8, '2019-02-14'),
(9, 9, '2015-08-30'),
(10, 10, '2020-05-18'),
(11, 11, '2017-10-03'),
(12, 12, '2018-12-11'),
(13, 13, '2016-06-22'),
(14, 14, '2019-09-07'),
(15, 15, '2015-03-29'),
(16, 16, '2021-01-15'),
(17, 17, '2017-04-08'),
(18, 18, '2018-11-19'),
(19, 19, '2016-02-26'),
(20, 20, '2020-07-13'),
(21, 21, '2015-12-05'),
(22, 22, '2019-05-21'),
(23, 23, '2017-08-14'),
(24, 24, '2021-03-02'),
(25, 25, '2018-10-27'),
(26, 26, '2016-01-09'),
(27, 27, '2020-06-16'),
(28, 28, '2019-11-24'),
(29, 29, '2015-04-11'),
(30, 30, '2022-01-28');

-- DOCTOR_AVAILABILITY TABLE

INSERT INTO Doctor_Availability (availability_id, doctor_id, slot_start_time, slot_end_time, is_available) VALUES
(1, 1, '2025-02-17 09:00:00', '2025-02-17 10:00:00', 'YES'),
(2, 1, '2025-02-17 10:00:00', '2025-02-17 11:00:00', 'YES'),
(3, 2, '2025-02-17 09:00:00', '2025-02-17 10:00:00', 'YES'),
(4, 2, '2025-02-17 14:00:00', '2025-02-17 15:00:00', 'NO'),
(5, 3, '2025-02-17 11:00:00', '2025-02-17 12:00:00', 'YES'),
(6, 3, '2025-02-17 13:00:00', '2025-02-17 14:00:00', 'YES'),
(7, 4, '2025-02-17 08:00:00', '2025-02-17 09:00:00', 'YES'),
(8, 4, '2025-02-17 15:00:00', '2025-02-17 16:00:00', 'NO'),
(9, 5, '2025-02-17 10:00:00', '2025-02-17 11:00:00', 'YES'),
(10, 5, '2025-02-17 16:00:00', '2025-02-17 17:00:00', 'YES'),
(11, 6, '2025-02-18 09:00:00', '2025-02-18 10:00:00', 'YES'),
(12, 6, '2025-02-18 11:00:00', '2025-02-18 12:00:00', 'NO'),
(13, 7, '2025-02-18 13:00:00', '2025-02-18 14:00:00', 'YES'),
(14, 7, '2025-02-18 14:00:00', '2025-02-18 15:00:00', 'YES'),
(15, 8, '2025-02-18 10:00:00', '2025-02-18 11:00:00', 'YES'),
(16, 8, '2025-02-18 15:00:00', '2025-02-18 16:00:00', 'NO'),
(17, 9, '2025-02-19 09:00:00', '2025-02-19 10:00:00', 'YES'),
(18, 9, '2025-02-19 11:00:00', '2025-02-19 12:00:00', 'YES'),
(19, 10, '2025-02-19 13:00:00', '2025-02-19 14:00:00', 'YES'),
(20, 10, '2025-02-19 14:00:00', '2025-02-19 15:00:00', 'NO'),
(21, 11, '2025-02-20 08:00:00', '2025-02-20 09:00:00', 'YES'),
(22, 11, '2025-02-20 16:00:00', '2025-02-20 17:00:00', 'YES'),
(23, 12, '2025-02-20 10:00:00', '2025-02-20 11:00:00', 'YES'),
(24, 12, '2025-02-20 15:00:00', '2025-02-20 16:00:00', 'NO'),
(25, 13, '2025-02-21 09:00:00', '2025-02-21 10:00:00', 'YES'),
(26, 13, '2025-02-21 14:00:00', '2025-02-21 15:00:00', 'YES'),
(27, 14, '2025-02-21 11:00:00', '2025-02-21 12:00:00', 'YES'),
(28, 14, '2025-02-21 13:00:00', '2025-02-21 14:00:00', 'NO'),
(29, 15, '2025-02-21 15:00:00', '2025-02-21 16:00:00', 'YES'),
(30, 15, '2025-02-21 16:00:00', '2025-02-21 17:00:00', 'YES');


 * sqlite:///hospital.db
90 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
50 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.
30 rows affected.


[]

In [24]:
%%sql

select * from user;

 * sqlite:///hospital.db
Done.


user_id,username,password,user_role,status
1,admin.johnson,hashed_pw_001,admin,active
2,admin.williams,hashed_pw_002,admin,active
3,admin.brown,hashed_pw_003,admin,active
4,admin.jones,hashed_pw_004,admin,active
5,admin.garcia,hashed_pw_005,admin,active
6,admin.miller,hashed_pw_006,admin,active
7,admin.davis,hashed_pw_007,admin,active
8,admin.rodriguez,hashed_pw_008,admin,active
9,admin.martinez,hashed_pw_009,admin,active
10,admin.hernandez,hashed_pw_010,admin,active


In [25]:
%%sql

select * from staff;

 * sqlite:///hospital.db
Done.


staff_id,user_id,first_name,last_name,phone,email,hire_date
1,61,Emma,Adams,555-0101,emma.adams@clinic.com,2020-03-15
2,62,Oliver,Baker,555-0102,oliver.baker@clinic.com,2019-07-22
3,63,Sophia,Carter,555-0103,sophia.carter@clinic.com,2021-01-10
4,64,Liam,Dixon,555-0104,liam.dixon@clinic.com,2018-11-05
5,65,Ava,Evans,555-0105,ava.evans@clinic.com,2022-02-14
6,66,Noah,Foster,555-0106,noah.foster@clinic.com,2020-09-30
7,67,Isabella,Green,555-0107,isabella.green@clinic.com,2019-04-18
8,68,Ethan,Hill,555-0108,ethan.hill@clinic.com,2021-06-25
9,69,Mia,Jenkins,555-0109,mia.jenkins@clinic.com,2020-12-08
10,70,Mason,King,555-0110,mason.king@clinic.com,2018-08-16


In [26]:
%%sql

select * from doctor;

 * sqlite:///hospital.db
Done.


doctor_id,user_id,first_name,last_name,phone,email,license_no,hire_date
1,31,Robert,Smith,555-0201,dr.smith@clinic.com,MD-2015-001234,2015-06-01
2,32,Priya,Patel,555-0202,dr.patel@clinic.com,MD-2016-002345,2016-03-15
3,33,Wei,Chen,555-0203,dr.chen@clinic.com,MD-2014-003456,2014-09-10
4,34,Min,Kim,555-0204,dr.kim@clinic.com,MD-2017-004567,2017-01-20
5,35,Linh,Nguyen,555-0205,dr.nguyen@clinic.com,MD-2015-005678,2015-11-08
6,36,Rajesh,Singh,555-0206,dr.singh@clinic.com,MD-2018-006789,2018-04-12
7,37,Fatima,Ahmed,555-0207,dr.ahmed@clinic.com,MD-2016-007890,2016-07-25
8,38,Sean,Murphy,555-0208,dr.murphy@clinic.com,MD-2019-008901,2019-02-14
9,39,Patrick,OConnor,555-0209,dr.oconnor@clinic.com,MD-2015-009012,2015-08-30
10,40,Maria,Rivera,555-0210,dr.rivera@clinic.com,MD-2020-010123,2020-05-18


In [ ]:
%%sql

select * from patient;

 * sqlite:///hospital.db
Done.


patient_id,first_name,last_name,date_of_birth,gender,phone,email,address,registration_date
1,Benjamin,Anderson,1985-03-12,Male,555-0301,b.anderson@email.com,"123 Oak Street, Dallas, TX 75201",2023-01-15
2,Emily,Martinez,1992-07-22,Female,555-0302,e.martinez@email.com,"456 Elm Avenue, Dallas, TX 75202",2023-02-20
3,Jacob,Thompson,1978-11-05,Male,555-0303,j.thompson@email.com,"789 Pine Road, Dallas, TX 75203",2023-03-10
4,Olivia,White,1988-04-18,Female,555-0304,o.white@email.com,"321 Maple Lane, Dallas, TX 75204",2023-04-05
5,William,Lopez,1995-09-30,Male,555-0305,w.lopez@email.com,"654 Birch Drive, Dallas, TX 75205",2023-05-12
6,Sophia,Lee,1983-12-08,Female,555-0306,s.lee@email.com,"987 Cedar Court, Dallas, TX 75206",2023-06-18
7,James,Walker,1990-02-14,Male,555-0307,j.walker@email.com,"147 Spruce Street, Dallas, TX 75207",2023-07-25
8,Ava,Hall,1987-06-25,Female,555-0308,a.hall@email.com,"258 Willow Way, Dallas, TX 75208",2023-08-30
9,Michael,Allen,1993-10-11,Male,555-0309,m.allen@email.com,"369 Ash Boulevard, Dallas, TX 75209",2023-09-14
10,Isabella,Young,1981-01-27,Female,555-0310,i.young@email.com,"741 Cherry Circle, Dallas, TX 75210",2023-10-22


In [27]:
%%sql
select * from appointment;

 * sqlite:///hospital.db
Done.


appointment_id,doctor_id,patient_id,appointment_datetime,status,reason_for_visit,notes
1,1,1,2025-01-15 09:00:00,completed,Annual physical examination,"Patient arrived on time, vitals normal"
2,2,2,2025-01-16 10:30:00,completed,Follow-up for hypertension,Blood pressure improved since last visit
3,3,3,2025-01-17 14:00:00,completed,Persistent cough and fever,Symptoms started 5 days ago
4,4,4,2025-01-18 11:00:00,completed,Diabetes management consultation,Reviewing recent lab results
5,5,5,2025-01-19 15:30:00,completed,Sports injury - knee pain,Injured during basketball game
6,6,6,2025-01-20 09:30:00,completed,Prenatal checkup - 20 weeks,"Second pregnancy, no complications"
7,7,7,2025-01-21 13:00:00,completed,Skin rash evaluation,Rash appeared 2 weeks ago on arms
8,8,8,2025-01-22 16:00:00,completed,Migraine headaches consultation,Frequency increasing over past month
9,9,9,2025-01-23 10:00:00,completed,Chest pain assessment,Intermittent pain for 3 days
10,10,10,2025-01-24 14:30:00,completed,Depression and anxiety screening,Feeling overwhelmed lately


In [ ]:
%%sql
select * from medical_record;

 * sqlite:///hospital.db
Done.


record_id,appointment_id,visit_date,diagnosis,visit_notes,treatment_plan,created_date,last_modified_date,lab_needed
1,1,2025-01-15,Healthy - no acute conditions,"BP: 120/80, HR: 72, Temp: 98.6F. All systems normal.",Continue healthy lifestyle. Return in 1 year for next physical.,2025-01-15 09:30:00,2025-01-15 09:30:00,YES
2,2,2025-01-16,Essential hypertension - controlled,"BP: 128/82, improved from 145/95 last month. Patient compliant with medication.",Continue Lisinopril 10mg daily. Follow up in 3 months.,2025-01-16 11:00:00,2025-01-16 11:00:00,NO
3,3,2025-01-17,Acute bronchitis,"Productive cough, low-grade fever (100.2F), lungs with rhonchi bilaterally.","Prescribe antibiotic, cough suppressant. Rest and fluids. Return if worsening.",2025-01-17 14:30:00,2025-01-17 14:30:00,NO
4,4,2025-01-18,Type 2 diabetes mellitus - well controlled,"HbA1c 6.8%, down from 7.4%. Patient monitoring glucose regularly.",Continue Metformin 1000mg BID. Maintain diet and exercise. Recheck labs in 3 months.,2025-01-18 11:30:00,2025-01-18 11:30:00,YES
5,5,2025-01-19,Medial meniscus tear - left knee,Swelling and tenderness over medial joint line. Positive McMurray test.,MRI ordered. RICE protocol. Refer to orthopedics. NSAIDs for pain.,2025-01-19 16:00:00,2025-01-19 16:00:00,NO
6,6,2025-01-20,"Pregnancy - 20 weeks, progressing normally","Fetal heart rate 145 bpm, fundal height appropriate. No complaints.",Continue prenatal vitamins. Anatomy ultrasound scheduled. Return in 4 weeks.,2025-01-20 10:00:00,2025-01-20 10:00:00,YES
7,7,2025-01-21,Contact dermatitis,Erythematous rash with vesicles on bilateral forearms. History suggests allergic reaction.,Topical corticosteroid cream. Avoid suspected allergen. Follow up in 2 weeks.,2025-01-21 13:30:00,2025-01-21 13:30:00,NO
8,8,2025-01-22,Chronic migraine without aura,Recurring headaches 15+ days/month. Photophobia and nausea present.,Start prophylactic medication. Keep headache diary. Avoid triggers. Follow up in 6 weeks.,2025-01-22 16:30:00,2025-01-22 16:30:00,NO
9,9,2025-01-23,Non-cardiac chest pain - GERD likely,"EKG normal, troponin negative. Pain associated with meals. No cardiac risk factors.",Trial of PPI therapy. Dietary modifications. Cardiology consult if symptoms persist.,2025-01-23 10:30:00,2025-01-23 10:30:00,YES
10,10,2025-01-24,"Major depressive disorder, moderate","PHQ-9 score: 16. Sleep disturbance, anhedonia, fatigue present for 6 weeks.",Start SSRI. Referral to therapist. Safety plan discussed. Follow up in 2 weeks.,2025-01-24 15:00:00,2025-01-24 15:00:00,NO


In [28]:
%%sql
select * from prescription;

 * sqlite:///hospital.db
Done.


prescription_id,record_id,prescription_date,medication_name,dosage,frequency,instructions,duration
1,2,2025-01-16,Lisinopril,10mg,Once daily,Take in the morning with or without food,90 days
2,3,2025-01-17,Azithromycin,250mg,Once daily,"Take on empty stomach, 1 hour before or 2 hours after meals",5 days
3,3,2025-01-17,Benzonatate,100mg,Three times daily,"Take with water, do not chew capsules",10 days
4,4,2025-01-18,Metformin,1000mg,Twice daily,Take with meals to reduce stomach upset,90 days
5,5,2025-01-19,Ibuprofen,600mg,Three times daily,"Take with food or milk, maximum 3 doses per day",14 days
6,6,2025-01-20,Prenatal Vitamins,One tablet,Once daily,Take with food for better absorption,30 days
7,7,2025-01-21,Hydrocortisone Cream 1%,Apply thin layer,Twice daily,"Apply to affected areas only, avoid face",14 days
8,8,2025-01-22,Topiramate,25mg,Once daily at bedtime,"Start low dose, may increase after 2 weeks. Drink plenty of water.",30 days
9,9,2025-01-23,Omeprazole,20mg,Once daily,Take 30 minutes before breakfast,30 days
10,10,2025-01-24,Sertraline,50mg,Once daily,Take in morning with food. May take 4-6 weeks for full effect.,30 days


In [29]:
%%sql
select * from billing;

 * sqlite:///hospital.db
Done.


billing_id,appointment_id,billing_date,total_amount,amount_paid,amount_due,payment_status
1,1,2025-01-15 10:00:00,250.0,250.0,0.0,paid
2,2,2025-01-16 11:30:00,150.0,150.0,0.0,paid
3,3,2025-01-17 15:00:00,200.0,50.0,150.0,partial
4,4,2025-01-18 12:00:00,325.0,325.0,0.0,paid
5,5,2025-01-19 16:30:00,275.0,0.0,275.0,pending
6,6,2025-01-20 10:30:00,180.0,180.0,0.0,paid
7,7,2025-01-21 14:00:00,165.0,165.0,0.0,paid
8,8,2025-01-22 17:00:00,220.0,100.0,120.0,partial
9,9,2025-01-23 11:00:00,450.0,450.0,0.0,paid
10,10,2025-01-24 15:30:00,200.0,200.0,0.0,paid


In [ ]:
%%sql
select * from billing_item;

 * sqlite:///hospital.db
Done.


item_id,billing_id,service_type,description,unit_cost,quantity,total_cost
1,1,consultation,Annual Physical Examination,150.0,1,150.0
2,1,lab,Complete Blood Count,50.0,1,50.0
3,1,lab,Lipid Panel,50.0,1,50.0
4,2,consultation,Follow-up Visit - Hypertension,150.0,1,150.0
5,3,consultation,Sick Visit - Respiratory,150.0,1,150.0
6,3,medication,Antibiotic Prescription,50.0,1,50.0
7,4,consultation,Diabetes Management Consultation,175.0,1,175.0
8,4,lab,HbA1c Test,75.0,1,75.0
9,4,lab,Glucose Fasting,75.0,1,75.0
10,5,consultation,Sports Medicine Consultation,200.0,1,200.0


In [30]:
%%sql
select * from insurance_policy;

 * sqlite:///hospital.db
Done.


policy_id,provider_name,policy_number,coverage_type,coverage_start_date,coverage_end_date,coverage_amount
1,Blue Cross Blue Shield,BCBS-2024-001234,PPO Premium,2024-01-01,2024-12-31,500000.0
2,United Healthcare,UHC-2024-005678,HMO Standard,2024-01-01,2024-12-31,300000.0
3,Aetna,AET-2024-009012,PPO Gold,2024-01-01,2024-12-31,750000.0
4,Cigna,CIG-2024-003456,EPO Plus,2024-02-01,2025-01-31,400000.0
5,Humana,HUM-2024-007890,PPO Basic,2024-01-01,2024-12-31,250000.0
6,Kaiser Permanente,KP-2024-001111,HMO Platinum,2024-03-01,2025-02-28,1000000.0
7,Blue Cross Blue Shield,BCBS-2024-002222,PPO Standard,2024-01-01,2024-12-31,350000.0
8,United Healthcare,UHC-2024-003333,PPO Premium,2024-04-01,2025-03-31,600000.0
9,Aetna,AET-2024-004444,HMO Basic,2024-01-01,2024-12-31,200000.0
10,Cigna,CIG-2024-005555,PPO Gold,2024-05-01,2025-04-30,550000.0


In [31]:
%%sql
select * from patient_insurance

 * sqlite:///hospital.db
Done.


patient_id,policy_id,relationship_type,effective_date
1,1,Self,2024-01-01
2,2,Self,2024-01-01
3,3,Self,2024-01-01
4,4,Self,2024-02-01
5,5,Self,2024-01-01
6,6,Self,2024-03-01
7,7,Self,2024-01-01
8,8,Self,2024-04-01
9,9,Self,2024-01-01
10,10,Self,2024-05-01


In [ ]:
%%sql
select * from insurance_claim;

 * sqlite:///hospital.db
Done.


claim_id,policy_id,billing_id,claim_date,claim_amount,approval_date,notes,status
1,1,1,2025-01-16 09:00:00,200.0,2025-01-20 14:30:00,Annual physical - approved with standard coverage,approved
2,2,2,2025-01-17 10:00:00,120.0,2025-01-21 11:00:00,Follow-up visit approved,approved
3,3,3,2025-01-18 11:00:00,150.0,2025-01-25 15:30:00,Sick visit claim approved,approved
4,4,4,2025-01-19 09:30:00,260.0,2025-01-23 10:00:00,Diabetes management approved,approved
5,5,5,2025-01-20 10:00:00,220.0,2025-02-01 09:00:00,Sports injury consultation pending review,under_review
6,6,6,2025-01-21 11:00:00,144.0,2025-01-26 13:00:00,Prenatal care approved - 80% coverage,approved
7,7,7,2025-01-22 12:00:00,132.0,2025-01-28 14:00:00,Dermatology consultation approved,approved
8,8,8,2025-01-23 13:00:00,176.0,2025-01-30 16:00:00,Neurology consultation approved,approved
9,9,9,2025-01-24 10:30:00,360.0,2025-01-27 11:30:00,Emergency consultation approved,approved
10,10,10,2025-01-25 14:00:00,160.0,2025-02-02 10:00:00,Mental health evaluation approved,approved


In [32]:
%%sql
select * from lab_test

 * sqlite:///hospital.db
Done.


test_id,test_name,test_description,standard_cost
1,Complete Blood Count (CBC),"Measures red cells, white cells, platelets, hemoglobin, and hematocrit",45.0
2,Basic Metabolic Panel (BMP),"Tests glucose, calcium, electrolytes, and kidney function",55.0
3,Comprehensive Metabolic Panel (CMP),Extended panel including liver function tests,75.0
4,Lipid Panel,"Cholesterol, HDL, LDL, and triglycerides measurement",60.0
5,Hemoglobin A1C (HbA1c),Average blood sugar over past 2-3 months,65.0
6,Thyroid Stimulating Hormone (TSH),Thyroid function screening test,50.0
7,Free T4 (Thyroxine),Thyroid hormone level measurement,55.0
8,Vitamin D 25-Hydroxy,Vitamin D level assessment,85.0
9,Vitamin B12,B12 deficiency screening,70.0
10,Urinalysis,Urine chemical and microscopic analysis,25.0


In [33]:
%%sql
select * from lab_report;

 * sqlite:///hospital.db
Done.


report_id,record_id,user_id,test_id,test_date,result,status,report_file_path,notes
1,1,61,1,2025-01-15,"WBC: 7.2, RBC: 4.8, Hgb: 14.2, Hct: 42%, Platelets: 245 - All values within normal range",completed,/lab_reports/2025/01/report_001_cbc.pdf,Normal complete blood count
2,1,61,4,2025-01-15,"Total Cholesterol: 185, HDL: 55, LDL: 105, Triglycerides: 125 - Optimal levels",reviewed,/lab_reports/2025/01/report_002_lipid.pdf,Excellent lipid profile
3,4,62,5,2025-01-18,HbA1c: 6.8% - Good diabetic control,completed,/lab_reports/2025/01/report_003_hba1c.pdf,Improved from previous 7.4%
4,4,62,2,2025-01-18,"Glucose: 118, BUN: 16, Creatinine: 0.9, eGFR: >90 - Normal kidney function",reviewed,/lab_reports/2025/01/report_004_bmp.pdf,Kidney function excellent
5,6,63,12,2025-01-20,"hCG: 45,200 mIU/mL - Consistent with 20 weeks gestation",completed,/lab_reports/2025/01/report_005_hcg.pdf,Pregnancy progressing normally
6,6,63,1,2025-01-20,"Hgb: 11.8, Hct: 35% - Mild anemia of pregnancy",reviewed,/lab_reports/2025/01/report_006_cbc.pdf,Iron supplementation recommended
7,9,64,23,2025-01-23,Troponin I: <0.01 ng/mL - Negative for cardiac injury,completed,/lab_reports/2025/01/report_007_troponin.pdf,Rules out myocardial infarction
8,9,64,1,2025-01-23,All blood counts within normal limits,reviewed,/lab_reports/2025/01/report_008_cbc.pdf,No abnormalities detected
9,15,65,4,2025-01-29,"Total Cholesterol: 225, LDL: 145, HDL: 48, Triglycerides: 160 - Improved on statin",completed,/lab_reports/2025/01/report_009_lipid.pdf,LDL decreased from 180
10,15,65,18,2025-01-29,"AST: 28, ALT: 32, Alk Phos: 78 - Liver function normal on statin",reviewed,/lab_reports/2025/01/report_010_lft.pdf,Safe to continue medication


In [34]:
%%sql
select * from specialization;

 * sqlite:///hospital.db
Done.


specialization_id,specialization_name,description
1,Internal Medicine,"Diagnosis and treatment of adult diseases, preventive care, and chronic disease management"
2,Pediatrics,"Medical care for infants, children, and adolescents including growth and development"
3,Cardiology,"Heart and cardiovascular system disorders, including hypertension and heart disease"
4,Dermatology,"Skin, hair, and nail conditions including acne, eczema, and skin cancer"
5,Orthopedics,"Musculoskeletal system including bones, joints, ligaments, tendons, and muscles"
6,Obstetrics and Gynecology,"Women's reproductive health, pregnancy, and childbirth"
7,Neurology,"Nervous system disorders including headaches, epilepsy, and stroke"
8,Psychiatry,"Mental health disorders, emotional problems, and behavioral issues"
9,Gastroenterology,"Digestive system disorders including stomach, intestines, liver, and pancreas"
10,Pulmonology,Respiratory system and lung diseases including asthma and COPD


In [35]:
%%sql
select * from doctor_specialization;

 * sqlite:///hospital.db
Done.


doctor_id,specialization_id,assigned_date
1,1,2015-06-01
2,2,2016-03-15
3,3,2014-09-10
4,4,2017-01-20
5,5,2015-11-08
6,6,2018-04-12
7,7,2016-07-25
8,8,2019-02-14
9,9,2015-08-30
10,10,2020-05-18


In [36]:
%%sql
select * from doctor_availability;

 * sqlite:///hospital.db
Done.


availability_id,doctor_id,slot_start_time,slot_end_time,is_available
1,1,2025-02-17 09:00:00,2025-02-17 10:00:00,YES
2,1,2025-02-17 10:00:00,2025-02-17 11:00:00,YES
3,2,2025-02-17 09:00:00,2025-02-17 10:00:00,YES
4,2,2025-02-17 14:00:00,2025-02-17 15:00:00,NO
5,3,2025-02-17 11:00:00,2025-02-17 12:00:00,YES
6,3,2025-02-17 13:00:00,2025-02-17 14:00:00,YES
7,4,2025-02-17 08:00:00,2025-02-17 09:00:00,YES
8,4,2025-02-17 15:00:00,2025-02-17 16:00:00,NO
9,5,2025-02-17 10:00:00,2025-02-17 11:00:00,YES
10,5,2025-02-17 16:00:00,2025-02-17 17:00:00,YES


In [37]:
# Business Rules
# 1. Patient Registration: A patient must have a unique Patient ID and a valid date of birth to be registered in the system.

%%sql
INSERT INTO Patient (patient_id, first_name, last_name, date_of_birth, gender, phone, email, address, registration_date) VALUES
(13, 'Benjamin', 'Anderson', '1985-03-12', 'Male', '555-0301', 'b.anderson@email.com', '123 Oak Street, Dallas, TX 75201', '2023-01-15')


 * sqlite:///hospital.db
(sqlite3.IntegrityError) UNIQUE constraint failed: Patient.patient_id
[SQL: INSERT INTO Patient (patient_id, first_name, last_name, date_of_birth, gender, phone, email, address, registration_date) VALUES
(13, 'Benjamin', 'Anderson', '1985-03-12', 'Male', '555-0301', 'b.anderson@email.com', '123 Oak Street, Dallas, TX 75201', '2023-01-15')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [38]:
%%sql
INSERT INTO Patient (patient_id, first_name, last_name, date_of_birth, gender, phone, email, address, registration_date) VALUES
(61, 'Benjamin', 'Anderson', '2027-03-12', 'Male', '555-0301', 'b.anderson@email.com', '123 Oak Street, Dallas, TX 75201', '2023-01-15')

 * sqlite:///hospital.db
(sqlite3.IntegrityError) CHECK constraint failed: date_of_birth > '1900-01-01' AND date_of_birth <= '2026-04-22'
[SQL: INSERT INTO Patient (patient_id, first_name, last_name, date_of_birth, gender, phone, email, address, registration_date) VALUES
(61, 'Benjamin', 'Anderson', '2027-03-12', 'Male', '555-0301', 'b.anderson@email.com', '123 Oak Street, Dallas, TX 75201', '2023-01-15')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [39]:
#2. Appointment Constraints: An appointment cannot be scheduled for a date and time that overlaps with another existing appointment for the same doctor.
#   A patient cannot have two active appointments scheduled for the same time.

%%sql
INSERT INTO Appointment (appointment_id, doctor_id, patient_id, appointment_datetime, status, reason_for_visit, notes) VALUES
(61, 1, 3, '2025-01-15 09:00:00', 'completed', 'Annual physical examination', 'Patient arrived on time, vitals normal')

 * sqlite:///hospital.db
(sqlite3.IntegrityError) UNIQUE constraint failed: Appointment.doctor_id, Appointment.appointment_datetime
[SQL: INSERT INTO Appointment (appointment_id, doctor_id, patient_id, appointment_datetime, status, reason_for_visit, notes) VALUES
(61, 1, 3, '2025-01-15 09:00:00', 'completed', 'Annual physical examination', 'Patient arrived on time, vitals normal')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [40]:
%%sql
INSERT INTO Appointment (appointment_id, doctor_id, patient_id, appointment_datetime, status, reason_for_visit, notes) VALUES
(61, 4, 1, '2025-01-15 09:00:00', 'completed', 'Annual physical examination', 'Patient arrived on time, vitals normal')

 * sqlite:///hospital.db
(sqlite3.IntegrityError) UNIQUE constraint failed: Appointment.patient_id, Appointment.appointment_datetime
[SQL: INSERT INTO Appointment (appointment_id, doctor_id, patient_id, appointment_datetime, status, reason_for_visit, notes) VALUES
(61, 4, 1, '2025-01-15 09:00:00', 'completed', 'Annual physical examination', 'Patient arrived on time, vitals normal')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [41]:
#3. Prescription Validity: A prescription must be associated with a valid patient and a valid doctor. It must include the date prescribed, the medication name, dosage, and instructions.

# Prescription table consists of all the mentioned fields and the patient_id and doctor_id can be retrieved by doing a join on the Medical_record table first on record_id.
# Following by an another join on the Appointment table on appointment_id.

In [42]:
#4. Insurance Policy Linkage: An insurance policy can be linked to one or more patients (e.g., a family plan).
#   A patient can be covered by more than one insurance policy (e.g., primary and secondary).

# There is no limitation kept on the Insurance_policy table that a patient should have a single record in that table. We can have any number of records with different policy_id's.

In [43]:
#5. Doctor Specialization: A doctor must be assigned to at least one specialization (e.g., Cardiology, Pediatrics) but can be assigned to more than one.

# All the doctors present in the Doctor table are been assigned with atleast one specialization in the doctor_specialization table.
# And we can insert any number of records for a single doctor into the doctor_specialization table which ensures that a doctor can have more than one specialization.

In [44]:
#6. Billing Integrity: A bill can only be generated after a patient visit has been marked as 'Completed' by the doctor.
#   A bill item (e.g., for a consultation or lab test) must be linked to a specific service or procedure.

# Bill item consists of the service_type enum NOT NULL field which ensures that every  entry in the table is tagged to a specific service
%%sql
INSERT INTO Billing_Item (item_id, billing_id, service_type, description, unit_cost, quantity, total_cost) VALUES
(101, 1, NULL, 'Annual Physical Examination', 150.00, 1, 150.00)

 * sqlite:///hospital.db
(sqlite3.IntegrityError) NOT NULL constraint failed: Billing_Item.service_type
[SQL: INSERT INTO Billing_Item (item_id, billing_id, service_type, description, unit_cost, quantity, total_cost) VALUES
(101, 1, NULL, 'Annual Physical Examination', 150.00, 1, 150.00)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [45]:
%%sql
INSERT INTO Billing_Item (item_id, billing_id, service_type, description, unit_cost, quantity, total_cost) VALUES
(101, 1, 'procedure1', 'Annual Physical Examination', 150.00, 1, 150.00)

 * sqlite:///hospital.db
(sqlite3.IntegrityError) CHECK constraint failed: service_type IN ('consultation', 'lab', 'medication', 'procedure')
[SQL: INSERT INTO Billing_Item (item_id, billing_id, service_type, description, unit_cost, quantity, total_cost) VALUES
(101, 1, 'procedure1', 'Annual Physical Examination', 150.00, 1, 150.00)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [47]:
#Queries

In [46]:
#1. List complete visit details including appointment_id, patient_name(concatenate first_name and last_name), doctor_name(concatenate first_name and last_name), diagnosis,
#   prescription(include the medication_name) and billing information(include the total_amount and payment_status) for all completed appointments. sort the data by latest appointment_datetime.

%%sql

SELECT
    a.appointment_id, p.first_name || ' ' || p.last_name AS patient_name, d.first_name || ' ' || d.last_name AS doctor_name,
    mr.diagnosis, pr.medication_name, b.total_amount, b.payment_status
FROM Appointment a
JOIN Patient p ON a.patient_id = p.patient_id
JOIN Doctor d ON a.doctor_id = d.doctor_id
JOIN Medical_Record mr ON a.appointment_id = mr.appointment_id
LEFT JOIN Prescription pr ON mr.record_id = pr.record_id
LEFT JOIN Billing b ON a.appointment_id = b.appointment_id
WHERE a.status = 'completed'
ORDER BY a.appointment_datetime DESC;

 * sqlite:///hospital.db
Done.


appointment_id,patient_name,doctor_name,diagnosis,medication_name,total_amount,payment_status
25,Andrew Roberts,Andrew Bennett,Acute cystitis,Nitrofurantoin,215.0,pending
24,Elizabeth Perez,Michelle Brooks,Influenza prophylaxis,None,85.0,paid
23,Christopher Mitchell,Christopher Watson,"Cholecystitis - chronic, pre-op clearance",Cephalexin,425.0,paid
22,Abigail Carter,Amanda Wood,Plantar fasciitis - right foot,Diclofenac Gel,155.0,paid
21,David Nelson,Thomas Richardson,Hypothyroidism - undertreated,Levothyroxine,265.0,partial
20,Evelyn Gonzalez,Carmen Diaz,Allergic rhinitis - seasonal,Cetirizine,335.0,paid
20,Evelyn Gonzalez,Carmen Diaz,Allergic rhinitis - seasonal,Fluticasone Nasal Spray,335.0,paid
19,Joseph Baker,Daniel Cox,Obstructive sleep apnea - suspected,None,190.0,paid
18,Harper Adams,Lisa Ward,Rheumatoid arthritis - active,Folic Acid,380.0,pending
18,Harper Adams,Lisa Ward,Rheumatoid arthritis - active,Methotrexate,380.0,pending


In [48]:
#2. For each doctor, show how many completed appointments resulted in billing, along with the minimum, maximum, average, and total billing amounts they generated.
#   Include doctor_id, doctor_name(concatenate first_name and last_name), total no.of billings as total_bills, minimum, maximum, average, and total billing amounts they generated as
#   min_bill, max_bill, avg_bill and total_revenue. Show the results sorted by total revenue in descending order.

%%sql

SELECT
    d.doctor_id, d.first_name || ' ' || d.last_name AS doctor_name,
    COUNT(b.billing_id) AS total_bills, MIN(b.total_amount) AS min_bill,
    MAX(b.total_amount) AS max_bill, AVG(b.total_amount) AS avg_bill, SUM(b.total_amount) AS total_revenue
FROM Doctor d
JOIN Appointment a ON d.doctor_id = a.doctor_id
JOIN Billing b ON a.appointment_id = b.appointment_id
WHERE a.status = 'completed'
GROUP BY d.doctor_id
ORDER BY total_revenue DESC;

 * sqlite:///hospital.db
Done.


doctor_id,doctor_name,total_bills,min_bill,max_bill,avg_bill,total_revenue
9,Patrick OConnor,1,450.0,450.0,450.0,450.0
23,Christopher Watson,1,425.0,425.0,425.0,425.0
18,Lisa Ward,1,380.0,380.0,380.0,380.0
20,Carmen Diaz,1,335.0,335.0,335.0,335.0
4,Min Kim,1,325.0,325.0,325.0,325.0
17,David Howard,1,300.0,300.0,300.0,300.0
15,Michael Reed,1,295.0,295.0,295.0,295.0
5,Linh Nguyen,1,275.0,275.0,275.0,275.0
21,Thomas Richardson,1,265.0,265.0,265.0,265.0
1,Robert Smith,1,250.0,250.0,250.0,250.0


In [49]:
#3. List patients, their doctors, insurance provider, and claim status for all billed appointments.
#   Include patient_name(concatenate first_name and last_name), doctor_name(concatenate first_name and last_name), provider_name, claim_amount and status of the claim as claim_status in the result.

%%sql

SELECT
    p.first_name || ' ' || p.last_name AS patient_name, d.first_name || ' ' || d.last_name AS doctor_name,
    ip.provider_name, ic.claim_amount, ic.status AS claim_status
FROM Patient p
JOIN Appointment a ON p.patient_id = a.patient_id
JOIN Doctor d ON a.doctor_id = d.doctor_id
JOIN Billing b ON a.appointment_id = b.appointment_id
JOIN Insurance_Claim ic ON b.billing_id = ic.billing_id
JOIN Insurance_Policy ip ON ic.policy_id = ip.policy_id;

 * sqlite:///hospital.db
Done.


patient_name,doctor_name,provider_name,claim_amount,claim_status
Benjamin Anderson,Robert Smith,Blue Cross Blue Shield,200.0,approved
Emily Martinez,Priya Patel,United Healthcare,120.0,approved
Jacob Thompson,Wei Chen,Aetna,150.0,approved
Olivia White,Min Kim,Cigna,260.0,approved
William Lopez,Linh Nguyen,Humana,220.0,under_review
Sophia Lee,Rajesh Singh,Kaiser Permanente,144.0,approved
James Walker,Fatima Ahmed,Blue Cross Blue Shield,132.0,approved
Ava Hall,Sean Murphy,United Healthcare,176.0,approved
Michael Allen,Patrick OConnor,Aetna,360.0,approved
Isabella Young,Maria Rivera,Cigna,160.0,approved


In [50]:
#4. Find doctors who have handled at least one completed appointment with billing records.
#   Include doctor_id, doctor_name(concatenate first_name and last_name), total no.of completed appointments as total_completed_appointments, and the average billing amount as avg_billing_amount.
#   Only include doctors whose average billing amount is greater than 300. Show the results sorted by avg_billing_amount in descending order.

%%sql

SELECT
    d.doctor_id, d.first_name || ' ' || d.last_name AS doctor_name,
    COUNT(a.appointment_id) AS total_completed_appointments,
    AVG(b.total_amount) AS avg_billing_amount
FROM Doctor d
JOIN Appointment a ON d.doctor_id = a.doctor_id
JOIN Billing b ON a.appointment_id = b.appointment_id
WHERE a.status = 'completed'
GROUP BY d.doctor_id
HAVING AVG(b.total_amount) > 300
ORDER BY avg_billing_amount DESC;

 * sqlite:///hospital.db
Done.


doctor_id,doctor_name,total_completed_appointments,avg_billing_amount
9,Patrick OConnor,1,450.0
23,Christopher Watson,1,425.0
18,Lisa Ward,1,380.0
20,Carmen Diaz,1,335.0
4,Min Kim,1,325.0


In [51]:
#5. Show patients with insurance coverage whose claims were denied, along with billing and doctor details.
#   Include the patient_name(concatenate first_name and last_name), doctor_name(concatenate first_name and last_name), provider_name, total_amount, claim_amount, status of the claim.

%%sql

SELECT
    p.first_name || ' ' || p.last_name AS patient_name, d.first_name || ' ' || d.last_name AS doctor_name,
    ip.provider_name, b.total_amount, ic.claim_amount, ic.status
FROM Patient p
JOIN Patient_Insurance pi ON p.patient_id = pi.patient_id
JOIN Insurance_Policy ip ON pi.policy_id = ip.policy_id
JOIN Insurance_Claim ic ON ip.policy_id = ic.policy_id
JOIN Billing b ON ic.billing_id = b.billing_id
JOIN Appointment a ON b.appointment_id = a.appointment_id
JOIN Doctor d ON a.doctor_id = d.doctor_id
WHERE ic.status = 'denied';

 * sqlite:///hospital.db
Done.


patient_name,doctor_name,provider_name,total_amount,claim_amount,status
Ryan Parker,Kevin Hughes,Humana,175.0,140.0,denied


In [52]:
%%sql
select * from doctor;

 * sqlite:///hospital.db
Done.


doctor_id,user_id,first_name,last_name,phone,email,license_no,hire_date
1,31,Robert,Smith,555-0201,dr.smith@clinic.com,MD-2015-001234,2015-06-01
2,32,Priya,Patel,555-0202,dr.patel@clinic.com,MD-2016-002345,2016-03-15
3,33,Wei,Chen,555-0203,dr.chen@clinic.com,MD-2014-003456,2014-09-10
4,34,Min,Kim,555-0204,dr.kim@clinic.com,MD-2017-004567,2017-01-20
5,35,Linh,Nguyen,555-0205,dr.nguyen@clinic.com,MD-2015-005678,2015-11-08
6,36,Rajesh,Singh,555-0206,dr.singh@clinic.com,MD-2018-006789,2018-04-12
7,37,Fatima,Ahmed,555-0207,dr.ahmed@clinic.com,MD-2016-007890,2016-07-25
8,38,Sean,Murphy,555-0208,dr.murphy@clinic.com,MD-2019-008901,2019-02-14
9,39,Patrick,OConnor,555-0209,dr.oconnor@clinic.com,MD-2015-009012,2015-08-30
10,40,Maria,Rivera,555-0210,dr.rivera@clinic.com,MD-2020-010123,2020-05-18
